# Answer-only (AO) 蒸留学生の学習ノートブック（Google Colab 用）

研究課題「CoT蒸留による数学能力向上は、教師のどの能力の転移によるものか」を検証するため、
公開 CoT 学生（`jasonrqh/*_Math-CoT-20k_lr5e-5_ep8_bs256`）と比較する **Answer-only (AO) 学生** を、
**公式実装 `verl.trainer.fsdp_sft_trainer_ours` を無変更のまま** 使って学習する。

| 役割 | 使用資産 |
| --- | --- |
| 教師 | `Qwen/Qwen3-32B`（記録のみ。再生成はしない） |
| 学生の学習データ | `jasonrqh/Math-CoT-20k`（20,480 行）。AO の target は `response` 列から抽出した教師の最終回答 `\boxed{...}` |
| Base / CoT 学生 | 設定セルの `SUPPORTED_MODELS` を参照（まず Qwen3 系列） |
| 学習コード | Fork `nshiki08/rethink_sft_generalization_AnsOnlyDistill`（参照元 commit `71a442ea…`） |

## 実行順

| # | セクション | 内容 | GPU |
| --- | --- | --- | --- |
| 0 | 認証 | Colab Secrets（`HF_TOKEN`, 任意で `WANDB_API_KEY`）を読む。値は表示しない | 不要 |
| 設定 | 設定セル | モデル・revision・保存先・実行モード・学習設定・探索候補・HF 設定をここで編集 | 不要 |
| 1 | 研究条件の確認 | 対応モデル、公式基準設定、AO target とプロンプトの関係 | 不要 |
| 2 | Fork の clone・環境構築 | revision 固定、`upstream` は参照専用、公式依存関係の pin、環境記録 | 不要 |
| 3 | データ取得・AO 抽出・全行監査 | 元 20,480 行・行順・`message`・`advantage` を維持し `teacher_answer` 列を追加 | 不要 |
| 4 | 長さ・target・loss mask・実効設定 | 公式 tokenizer / chat template / dataset クラスで確認。`data.max_length` の提示 | 不要 |
| 5 | ドライラン（既定） | 起動コマンド・想定 step 数・保存回数・HF 保存先を確認。学習も HF 書き込みも開始しない | 不要 |
| 6 | 短い試走 | 少数 step の学習 → HF 保存 → プロセス終了 → HF から取得 → 新プロセスで再開 | 必要 |
| 7 | baseline | 公式 CoT 設定（lr 5e-5, 8 epoch）の AO 学習 | 必要 |
| 8 | LR / epoch 探索 | 候補ごとに同じ Base から独立に学習 | 必要 |
| 再開 | 切断後の再開 | HF の完了済み checkpoint 一覧 → 選択 → 検証 → `resume_path` で再開 | 必要 |
| 9 | dev 評価・集計 | 独立 dev での比較（dev 未確定なら選択処理の準備に留める） | 必要 |
| 10 | 最終モデル | `verl.model_merger` で HF 形式へ変換、HF 保存、読み込み・生成確認、Model Card | 必要 |

## 注意

- 既定の `RUN_MODE` は `"dry_run"`。学習と HF 書き込みは `"trial"` / `"train"` にしないと始まらない。
- Colab の切断でプロセスは消える。学習中の checkpoint は監視スレッドが保存完了ごとに HF へ転送し、過去 step は消さない。
- 公式実装（`verl/`, `training_scripts/`）は変更しない。追加処理はすべてこのノートブックにある。
- `git push` 先は Fork のみ。オリジナル (`Nebularaid2000/...`) は fetch 専用で push URL を無効化する。
- このノートブックの静的検証は CPU 環境で行った。GPU を要する項目（試走・学習・再開・変換）は各セルの実行ログが確認根拠になる。未実行の項目を「動作確認済み」とは書かない。

In [ ]:
# @title 0. 認証（最初のセル）: Colab Secrets / 環境変数からトークンを読む。値は表示しない
import os, sys, subprocess


def _get_secret(name):
    value = None
    try:
        from google.colab import userdata  # type: ignore

        try:
            value = userdata.get(name)
        except Exception:
            value = None
    except ImportError:
        pass
    if not value:
        value = os.environ.get(name)
    return value or None


_hf_token = _get_secret("HF_TOKEN")
_wandb_key = _get_secret("WANDB_API_KEY")

HF_LOGGED_IN = False
HF_ACCOUNT_NAME = None  # HF の whoami から取る。GitHub のユーザー名からは推測しない

# huggingface_hub はこのカーネルで最初に import する前に、公式 requirements.txt と同じ版へ揃える（import 後の pip では差し替わらない）
HUGGINGFACE_HUB_PIN = "0.34.4"
try:
    from importlib.metadata import version as _pkg_version
    _hub_installed = _pkg_version("huggingface_hub")
except Exception:
    _hub_installed = None
if _hub_installed != HUGGINGFACE_HUB_PIN:
    assert "huggingface_hub" not in sys.modules, "huggingface_hub が既に import されている。ランタイムを再起動してこのセルから実行する"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"huggingface_hub=={HUGGINGFACE_HUB_PIN}"])
import huggingface_hub
print("huggingface_hub", huggingface_hub.__version__)

if _hf_token:
    os.environ["HF_TOKEN"] = _hf_token
    huggingface_hub.login(token=_hf_token, add_to_git_credential=False)
    _who = huggingface_hub.whoami()
    HF_ACCOUNT_NAME = _who.get("name")
    HF_LOGGED_IN = True
    print(f"HF: logged in as '{HF_ACCOUNT_NAME}' (token masked)")
else:
    print("HF: HF_TOKEN が見つからない。HF へのアップロード・再開は無効。Colab Secrets に HF_TOKEN を登録して再実行する")

if _wandb_key:
    os.environ["WANDB_API_KEY"] = _wandb_key
    print("WANDB: API key found (masked)")
else:
    print("WANDB: API key なし。公式スクリプトと同じ offline mode を使う")

del _hf_token, _wandb_key

In [ ]:
# @title 設定セル（編集はここだけ）
import os, sys, json, re, time, datetime, pathlib

# =============================================================================
# 実行モード
#   dry_run : データ・設定・コマンド・想定 step 数を確認するだけ。学習も HF 書き込みも行わない（既定）
#   trial   : 少数 step の学習で、メモリ・保存・HF 転送・プロセス終了→再開を確認する
#   train   : baseline または探索候補を所定の条件で学習する
# =============================================================================
RUN_MODE = "dry_run"          # "dry_run" | "trial" | "train"
RUN_KIND = "baseline"         # train 時に "baseline" | "search"
RUN_ID_SUFFIX = ""            # 同名 run と衝突する場合に付ける識別子（例 "-v2"）。空なら付けない

# =============================================================================
# 対象モデル（まず Qwen3 系列。サイズはここで選ぶ）
#   1.7B: 小規模な動作確認候補 / 4B: 本実験候補（Colab で実行可能かは試走で実測する）
# =============================================================================
MODEL_KEY = "Qwen3-1.7B"

SUPPORTED_MODELS = {
    # revision は 2026-09-24 に HF API で確認した commit sha。ノートブック実行時にも再記録する
    "Qwen3-1.7B": dict(status="supported",
                       base_repo="Qwen/Qwen3-1.7B-Base", base_revision="ea980cb0a6c2ae4b936e82123acc929f1cec04c1",
                       cot_repo="jasonrqh/Qwen3-1.7B_Math-CoT-20k_lr5e-5_ep8_bs256", cot_revision="e4fcd58c7b104d3444e0e1d1b470f5107fcc2618",
                       cot_subfolder="step640", official_script="Qwen3-1.7B_Math-CoT-20k_lr5e-5_ep8_bs256.sh", note="小規模な動作確認候補"),
    "Qwen3-4B": dict(status="supported",
                     base_repo="Qwen/Qwen3-4B-Base", base_revision="906bfd4b4dc7f14ee4320094d8b41684abff8539",
                     cot_repo="jasonrqh/Qwen3-4B_Math-CoT-20k_lr5e-5_ep8_bs256", cot_revision="aaf1c51a6a2ee319750cafbd7cb8be0df10018b8",
                     cot_subfolder="step640", official_script="Qwen3-4B_Math-CoT-20k_lr5e-5_ep8_bs256.sh", note="本実験候補（Colab で収まるかは要実測）"),
    "Qwen3-8B": dict(status="supported",
                     base_repo="Qwen/Qwen3-8B-Base", base_revision="49e3418fbbbca6ecbdf9608b4d22e5a407081db4",
                     cot_repo="jasonrqh/Qwen3-8B_Math-CoT-20k_lr5e-5_ep8_bs256", cot_revision="522decd73c0b46f7029c2a315b3cc177c23a3498",
                     cot_subfolder="step640", official_script="Qwen3-8B_Math-CoT-20k_lr5e-5_ep8_bs256.sh", note="Colab 単一 GPU では full SFT が収まらない可能性が高い"),
    "Qwen3-14B": dict(status="supported",
                      base_repo="Qwen/Qwen3-14B-Base", base_revision="0b0bd3732e2c374d483664439ea334928b65f304",
                      cot_repo="jasonrqh/Qwen3-14B_Math-CoT-20k_lr5e-5_ep8_bs256", cot_revision="e8e6542249d647ac82c556a93410a9c0cafd41cb",
                      cot_subfolder="step640", official_script="Qwen3-14B_Math-CoT-20k_lr5e-5_ep8_bs256.sh", note="Colab 単一 GPU では full SFT が収まらない可能性が高い"),
    # Qwen2.5 系列: 公式スクリプトは Qwen/Qwen2.5-{size}-no-sys-prompt を指定する（HF API は 401 = 非公開または存在しない）。
    # 公開 Base の chat template は system message が無いと "You are a helpful assistant." を自動挿入するため tokenization が公式と一致しない。
    # 関係・準備方法を確認するまで黙って置き換えない → blocked。
    "Qwen2.5-1.5B": dict(status="blocked", base_repo="Qwen/Qwen2.5-1.5B", base_revision="8faed761d45a263340a0528343f099c05c9a4323",
                         cot_repo="jasonrqh/Qwen2.5-1.5B_Math-CoT-20k_lr5e-5_ep8_bs256", cot_revision="dd3c28b7846607e026f4ecc48eeb20e8d48035f1",
                         cot_subfolder=None, official_script="Qwen2.5-1.5B_Math-CoT-20k_lr5e-5_ep8_bs256.sh",
                         note="公式は Qwen/Qwen2.5-1.5B-no-sys-prompt（非公開）。公開 Base との差（chat template の既定 system prompt）が未解決"),
    "InternLM2.5-20B": dict(status="blocked", base_repo="internlm/internlm2_5-20b", base_revision="dc786ab20a7fd882ccef61df28e13a940edca24f",
                            cot_repo="jasonrqh/InternLM2.5-20B_Math-CoT-20k_lr5e-5_ep8_bs256", cot_revision="fa6f45a03525bb5a52046e3217f4ea5082aa4c75",
                            cot_subfolder=None, official_script="InternLM2.5-20B_Math-CoT-20k_lr5e-5_ep8_bs256.sh",
                            note="今回の対象外（trust_remote_code モデル。公式 micro_bsz=2）"),
}
TEACHER = dict(repo="Qwen/Qwen3-32B", revision="9216db5781bf21249d130ec9da846c4624c16137")  # 記録用。ダウンロードしない

# =============================================================================
# コードとデータの revision
# =============================================================================
FORK_REPO_URL = "https://github.com/nshiki08/rethink_sft_generalization_AnsOnlyDistill"
FORK_REF = "main"                 # ブランチ名または commit。実行時に HEAD の commit を記録する
UPSTREAM_REPO_URL = "https://github.com/Nebularaid2000/rethink_sft_generalization"   # 参照専用（push 無効化）
UPSTREAM_REFERENCE_COMMIT = "71a442ea8f0adc4a1df4529d3c43393ac6e504fd"               # 再現用の参照 commit

DATASET_REPO = "jasonrqh/Math-CoT-20k"
DATASET_REVISION = "1435fb21d4fecc8ad4966a26f22a874cf2b527f1"
DATASET_FILE = "Math-CoT-20k.parquet"
DATASET_EXPECTED_SHA256 = "3166f342ec838ebe0979be43923abb33fa1c80f33043aa6acc04e5d82161142a"
DATASET_EXPECTED_ROWS = 20480
DATASET_ORIGIN_POOL = "open-r1/OpenR1-Math-220k"   # 元問題プール（記録のみ。DeepSeek-R1 出力は抽出元にしない）

# =============================================================================
# AO データの抽出
#   teacher_answer 列 = response（教師 Qwen3-32B の CoT+回答）から抽出した最終回答
#   last_boxed_verbatim（既定）: 最後の \boxed{...} を教師の原文のまま保持（CoT・説明文は含めない）
#   boxed_content            : \boxed{} の中身のみ（教師原文の形を崩すので既定にしない）
#   last_boxed_line          : 最後の \boxed{} を含む原文の行（説明文が混ざり得るので既定にしない）
# =============================================================================
AO_TARGET_STYLE = "last_boxed_verbatim"
# math-verify で参照正解 (answer 列) と一致しなかった行のうち、内容を目視確認して「教師の最終回答として保持する」と決めた行番号。
# 2026-09-24 の監査では 3 行。いずれも同値な方程式の別表記（例 x^2/5 + 3y^2/5 = 1 と x^2+3y^2=5）で math-verify の関係式比較が失敗する。
# 実際の不一致集合がこのリストと異なれば本学習は開始しない（未解決の抽出問題として扱う）。
AO_ACKNOWLEDGED_VERIFY_MISMATCH_ROWS = [12364, 16249, 16611]

# =============================================================================
# 学習設定
#   公式基準（公開 CoT スクリプト）: AdamW β=(0.9,0.999), lr 5e-5, 8 epoch, global batch 256, cosine + warmup 10%,
#   weight decay 0.01, grad clip 1.0, bf16, max_length 20000, truncation right, shuffle False, token-mean, grad ckpt, FSDP
#   → 実際の値は公式スクリプトと sft_trainer.yaml を実行時に読み取って表示する（ここに書いた値は目安）
# =============================================================================
OFFICIAL_BASELINE = {"lr": 5e-5, "epochs": 8, "status": "confirmed: 公開 CoT スクリプトの値"}

# 探索対象は optim.lr と trainer.total_epochs。提案値と確定値を分ける。
SEARCH_GRID_PROPOSED = {"lr": [1e-5, 2e-5, 5e-5, 1e-4], "epochs": [1, 2, 4, 8]}   # 提案（未確定）
SEARCH_GRID_CONFIRMED = None      # 教授と合意後に {"lr": [...], "epochs": [...]} を入れる
SEARCH_RUN_LIST = []              # このセッションで実行する候補 [(lr, epochs), ...]。空なら探索セルは何もしない

# 最大系列長: "auto_fit" = 全行が切り詰められずに収まる最小の長さ（MAX_LENGTH_ROUND_TO の倍数に切り上げ）
#             "official" = 公式の 20000 / 整数 = 明示指定
MAX_LENGTH_MODE = "auto_fit"
MAX_LENGTH_ROUND_TO = 64

MICRO_BATCH_OVERRIDE = None       # None = 公式スクリプトのモデル別 micro batch を使う。整数で上書き（global batch 256 は維持）
CPU_OFFLOAD_OVERRIDE = None       # None = 公式値 (False)。True にすると model.fsdp_config.cpu_offload/offload_params を有効化（既存機能）
KEEP_OFFICIAL_ENV_VARS = True     # 公式スクリプトの export（CUDA_LAUNCH_BLOCKING=1 など）をそのまま使う。False なら CUDA_LAUNCH_BLOCKING を外し記録する

SAVE_FREQ = 40                    # trainer.save_freq（正の整数。0 は使わない）。公式 CoT は 10（=64 個/8epoch）。
                                  # checkpoint 1 個 ≈ bf16 params x3（AdamW 2 状態込み）: 1.7B ≈ 12 GB, 4B ≈ 24 GB。16 個で 190-390 GB。HF の容量上限に注意（ドライランで見積を表示）
SEED = None                       # None = 公式既定 (trainer.seed=1 は yaml 既定。trainer 内で明示的な乱数初期化はされていない)

# =============================================================================
# Hugging Face 保存
# =============================================================================
HF_CKPT_REPO_ID = None            # None → f"{HF_ACCOUNT_NAME}/rethink-sft-ao-checkpoints"（HF の whoami から決める）
HF_CKPT_PRIVATE = True
HF_UPLOAD_CHECKPOINTS = True      # 学習中に保存完了した checkpoint を監視スレッドが HF へ転送する
CKPT_MANIFEST_SHA256 = True       # manifest に sha256 を記録し、転送後に HF 側の LFS sha256 と照合する
LOCAL_KEEP_LAST_N_VERIFIED_CKPTS = 2   # HF 転送・検証済みのローカル checkpoint をこの数だけ残し、古いものを削除（未検証は削除しない）
HF_FINAL_MODEL_REPO_ID = None     # None → f"{HF_ACCOUNT_NAME}/{run_id}"（推論用 HF モデル）
HF_FINAL_MODEL_PRIVATE = True

WANDB_MODE = "offline"            # 公式スクリプトと同じ offline。"online" は WANDB_API_KEY がある場合のみ
WANDB_PROJECT = "sft_generalization_ao"

# =============================================================================
# 試走（RUN_MODE="trial"）
#   AO データの先頭 TRIAL_NUM_ROWS 行だけを使い、TRIAL_EPOCHS epoch 学習する。
#   steps_per_epoch = TRIAL_NUM_ROWS // 256。TRIAL_KILL_AFTER_STEP の checkpoint が HF へ転送・検証された時点でプロセスを終了し、
#   新しいプロセスで HF から取得して再開する。試走の run_id は "trial-" で始まり、本学習と混ざらない。
# =============================================================================
TRIAL_NUM_ROWS = 512
TRIAL_EPOCHS = 4                  # 2 steps/epoch x 4 = 8 step。kill step の後に十分な step が残るようにする
TRIAL_SAVE_FREQ = 3
TRIAL_KILL_AFTER_STEP = 3         # この step の checkpoint 保存完了を検知したら学習プロセスを一時停止し、HF 転送・検証後に終了する
TRIAL_RUN_REFERENCE = True        # 中断なしの参照 run も実行し、再開後の lr / loss / データ位置を比較する

# =============================================================================
# 切断後の再開（「再開」セルで使う）
# =============================================================================
RESUME_RUN_ID = None              # 例 "Qwen3-1.7B_Math-AO-20k_lr5e-5_ep8_bs256_baseline"
RESUME_STEP = "latest"            # "latest"（最新の完了済み）または整数 step

# =============================================================================
# dev 評価（セクション 9）。dev は未確定なので既定は無効。学習データや最終 test で選ばない
# =============================================================================
DEV_EVAL_ENABLED = False
DEV_EVAL_IS_INDEPENDENT = False   # dev が学習データ (Math-CoT-20k / OpenR1 由来の同一問題) とも最終 test とも重ならないと確認したら True にする
DEV_EVAL_SOURCE = None            # jsonl のパス、または HF dataset id（例 "org/name"）
DEV_EVAL_SPLIT = "test"
DEV_EVAL_REVISION = None
DEV_EVAL_QUESTION_KEY = "problem"
DEV_EVAL_ANSWER_KEY = "answer"
DEV_EVAL_MAX_ROWS = None
DEV_EVAL_MAX_NEW_TOKENS = 512     # AO 学生は短い出力。CoT 学生や Base を評価するときは大きくする
DEV_EVAL_BATCH_SIZE = 16
DEV_EVAL_DO_SAMPLE = False        # 公式評価 (math_eval_budget.py) は temperature 0.6 / top_p 0.95 / n サンプル。dev 選択は greedy を既定にし差を記録する
DEV_EVAL_TARGETS = []             # [{"name": "...", "path_or_repo": "...", "subfolder": None, "revision": None}, ...]

# =============================================================================
# 最終モデル（セクション 10）
# =============================================================================
FINAL_RUN_ID = None               # None → このセッションで学習した run
FINAL_STEP = "last"               # "last" または整数 step

# =============================================================================
# 作業ディレクトリと環境の pin
# =============================================================================
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
WORK_DIR = "/content/ao_work" if IN_COLAB else os.path.abspath("./ao_work")
REPO_DIR = f"{WORK_DIR}/repo"
DATA_DIR = f"{WORK_DIR}/data"
MODELS_DIR = f"{WORK_DIR}/models"
CKPT_DIR = f"{WORK_DIR}/ckpt"
LOG_DIR = f"{WORK_DIR}/log"
RECORD_DIR = f"{WORK_DIR}/records"
for _d in (WORK_DIR, DATA_DIR, MODELS_DIR, CKPT_DIR, LOG_DIR, RECORD_DIR):
    os.makedirs(_d, exist_ok=True)

# 公式 requirements.txt の pin から、学習に必要な範囲を抜き出したもの（vllm/sglang/評価系は含めない）
PIP_PINNED = [
    "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0",
    "transformers==4.52.4", "tokenizers==0.21.4", "accelerate==1.10.1", "datasets==4.0.0",
    "tensordict==0.9.1", "torchdata==0.11.0", "peft==0.17.1",
    "hydra-core==1.3.2", "omegaconf==2.3.0", "wandb==0.21.1", "ray[default]==2.43.0",
    "codetiming==1.4.0", "dill==0.3.8", "pyarrow==21.0.0", "numpy==1.26.4",
    "math-verify==0.7.0", "latex2sympy2_extended==1.10.1", "pylatexenc==2.10",
    "huggingface_hub==0.34.4", "hf-xet==1.1.9", "safetensors==0.6.2", "einops==0.8.1", "sentencepiece==0.2.1",
    "regex==2025.7.34", "word2number==1.1",   # evaluation/math_eval/utils/parser.py（既存の評価 parser）が使う
]
FLASH_ATTN_VERSION = "2.7.4.post1"   # 公式 pin。GitHub Releases の prebuilt wheel を torch/CUDA/Python/ABI に合わせて選ぶ
EXTRA_PIP_PACKAGES = []               # import 確認で不足が出た場合に追加

# ---- 設定の妥当性 ----
assert RUN_MODE in ("dry_run", "trial", "train"), RUN_MODE
assert RUN_KIND in ("baseline", "search"), RUN_KIND
assert isinstance(SAVE_FREQ, int) and SAVE_FREQ > 0, "trainer.save_freq は正の整数。0 は使わない"
assert isinstance(TRIAL_SAVE_FREQ, int) and TRIAL_SAVE_FREQ > 0
assert MAX_LENGTH_MODE in ("auto_fit", "official") or isinstance(MAX_LENGTH_MODE, int)
assert AO_TARGET_STYLE in ("last_boxed_verbatim", "boxed_content", "last_boxed_line")
assert MODEL_KEY in SUPPORTED_MODELS, f"未対応の MODEL_KEY: {MODEL_KEY}"
MODEL_INFO = SUPPORTED_MODELS[MODEL_KEY]

SESSION_STARTED_AT = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
print("RUN_MODE:", RUN_MODE, "| RUN_KIND:", RUN_KIND, "| MODEL_KEY:", MODEL_KEY, "| status:", MODEL_INFO["status"])
print("WORK_DIR:", WORK_DIR)

## 1. 設定・研究条件・対応モデルの確認

In [ ]:
# @title 1. 研究条件・対応モデルの確認
print("=== 対応モデル ===")
for _k, _v in SUPPORTED_MODELS.items():
    print(f"  {_k:16s} {_v['status']:9s} base={_v['base_repo']}@{_v['base_revision'][:8]}  cot={_v['cot_repo']}"
          f"{'/' + _v['cot_subfolder'] if _v['cot_subfolder'] else ''}@{_v['cot_revision'][:8]}")
    print(f"  {'':16s} note: {_v['note']}")
if MODEL_INFO["status"] != "supported":
    raise SystemExit(f"MODEL_KEY={MODEL_KEY} は blocked: {MODEL_INFO['note']}。設定を確認するまで進めない")

print("\n=== 比較対象 ===")
print(f"  Teacher            : {TEACHER['repo']} @ {TEACHER['revision'][:8]}（記録のみ）")
print(f"  Base Student       : {MODEL_INFO['base_repo']} @ {MODEL_INFO['base_revision'][:8]}")
print(f"  CoT Distill Student: {MODEL_INFO['cot_repo']}/{MODEL_INFO['cot_subfolder']} @ {MODEL_INFO['cot_revision'][:8]}（公開モデル。学習しない）")
print(f"  AO Distill Student : このノートブックで学習（Base から Full-parameter SFT）")

print("\n=== 公式基準設定（公開 CoT スクリプトを実行時に解析して表示する。ここは目安） ===")
print(f"  lr={OFFICIAL_BASELINE['lr']}, epochs={OFFICIAL_BASELINE['epochs']}  [{OFFICIAL_BASELINE['status']}]")
print("  探索候補（提案・未確定）:", SEARCH_GRID_PROPOSED)
print("  探索候補（確定）      :", SEARCH_GRID_CONFIRMED)
print("  このセッションで実行する候補:", SEARCH_RUN_LIST)

print("\n=== AO target とプロンプトの関係（実験記録に明記する） ===")
print("  入力 prompt (message 列) は公式と同一。末尾の『Please reason step by step, and put your final answer within \\boxed{}.』も変更しない。")
print("  target (teacher_answer 列) は教師 response の最後の \\boxed{...} を原文のまま用いる。CoT・説明文は含めない。")
print("  したがって AO 学生は『step by step』と指示されても推論を書かず \\boxed{答え} + EOS を出力するよう学習される。")
print("  公式の Math-NoCoT-20k（<think> を除き最終まとめ＋回答を残す）とは異なる。参照正解 answer 列は照合にだけ使い、target に代用しない。")

## 2. Fork の clone・revision 固定・環境構築

In [ ]:
# @title 2. Fork の clone・revision 固定・環境構築（公式依存関係を起点。Docker は使わない）
import subprocess, shutil, json, os, sys, platform, re, time


def sh(cmd, cwd=None, check=True, capture=True, env=None):
    """subprocess の薄いラッパ。失敗時はコマンドと出力を表示する"""
    r = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str), capture_output=capture, text=True, env=env)
    if check and r.returncode != 0:
        print("COMMAND FAILED:", cmd)
        print(r.stdout[-4000:] if r.stdout else "", r.stderr[-4000:] if r.stderr else "")
        raise RuntimeError(f"command failed ({r.returncode})")
    return (r.stdout or "").strip()


# ---- 2.1 Fork の clone と remote 設定 -------------------------------------------------------
if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    print("cloning", FORK_REPO_URL, "->", REPO_DIR)
    sh(["git", "clone", FORK_REPO_URL, REPO_DIR])
else:
    print("existing clone found at", REPO_DIR)
_origin = sh(["git", "remote", "get-url", "origin"], cwd=REPO_DIR)
assert _origin.rstrip("/").removesuffix(".git") == FORK_REPO_URL.rstrip("/").removesuffix(".git"), f"origin が Fork ではない: {_origin}"

_remotes = sh(["git", "remote"], cwd=REPO_DIR).split()
if "upstream" not in _remotes:
    sh(["git", "remote", "add", "upstream", UPSTREAM_REPO_URL], cwd=REPO_DIR)
sh(["git", "remote", "set-url", "--push", "upstream", "DISABLED_no_push_to_upstream"], cwd=REPO_DIR)   # オリジナルへの push を無効化
print("remotes:")
print(sh(["git", "remote", "-v"], cwd=REPO_DIR))
assert "DISABLED_no_push_to_upstream" in sh(["git", "remote", "get-url", "--push", "upstream"], cwd=REPO_DIR)

sh(["git", "fetch", "--quiet", "origin"], cwd=REPO_DIR)
try:
    sh(["git", "fetch", "--quiet", "upstream"], cwd=REPO_DIR)
    UPSTREAM_FETCHED = True
except RuntimeError:
    UPSTREAM_FETCHED = False
    print("WARN: upstream の fetch に失敗（参照 commit との差分確認は Fork 内の履歴で行う）")

# FORK_REF: origin のブランチ名なら origin/<ref>、そうでなければ commit として checkout（detached）
try:
    _target = sh(["git", "rev-parse", "--verify", f"origin/{FORK_REF}^{{commit}}"], cwd=REPO_DIR)
except RuntimeError:
    _target = sh(["git", "rev-parse", "--verify", f"{FORK_REF}^{{commit}}"], cwd=REPO_DIR)
sh(["git", "checkout", "--quiet", "--detach", _target], cwd=REPO_DIR)
FORK_COMMIT = sh(["git", "rev-parse", "HEAD"], cwd=REPO_DIR)
print("Fork HEAD:", FORK_COMMIT, "(ref:", FORK_REF + ")")
print("Upstream reference commit:", UPSTREAM_REFERENCE_COMMIT)

# 公式実装との差分（verl/ と training_scripts/ に変更が無いことを確認する）
try:
    _is_anc = subprocess.run(["git", "merge-base", "--is-ancestor", UPSTREAM_REFERENCE_COMMIT, "HEAD"], cwd=REPO_DIR).returncode == 0
    OFFICIAL_DIFF_STAT = sh(["git", "diff", "--stat", UPSTREAM_REFERENCE_COMMIT, "HEAD", "--", "verl", "training_scripts"], cwd=REPO_DIR)
    print("参照 commit は HEAD の祖先か:", _is_anc)
    print("公式コード (verl/, training_scripts/) と参照 commit の差分:", "なし" if not OFFICIAL_DIFF_STAT else "\n" + OFFICIAL_DIFF_STAT)
    OFFICIAL_CODE_UNCHANGED = (OFFICIAL_DIFF_STAT == "")
except RuntimeError:
    OFFICIAL_DIFF_STAT, OFFICIAL_CODE_UNCHANGED = "unknown (reference commit not available)", None
    print("WARN: 参照 commit がローカルに無く差分を確認できない")

# ---- 2.2 ハードウェア情報 -------------------------------------------------------------------
def gpu_info():
    if shutil.which("nvidia-smi") is None:
        return []
    out = sh(["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version,compute_cap", "--format=csv,noheader,nounits"], check=False)
    gpus = []
    for line in out.splitlines():
        parts = [p.strip() for p in line.split(",")]
        if len(parts) >= 3:
            gpus.append(dict(index=int(parts[0]), name=parts[1], vram_mb=int(float(parts[2])),
                             driver=parts[3] if len(parts) > 3 else None, compute_cap=parts[4] if len(parts) > 4 else None))
    return gpus


GPUS = gpu_info()
N_GPUS = len(GPUS)
_mem = {}
for line in open("/proc/meminfo"):
    k, v = line.split(":")[0], line.split(":")[1].strip().split()[0]
    _mem[k] = int(v)
_disk = shutil.disk_usage(WORK_DIR)
print(f"GPU x{N_GPUS}:", GPUS)
print(f"RAM total {_mem.get('MemTotal', 0) / 1e6:.1f} GB | disk free {_disk.free / 1e9:.1f} GB / total {_disk.total / 1e9:.1f} GB | python {platform.python_version()}")
if N_GPUS == 0:
    print("WARN: GPU が無い。dry_run とデータ処理のみ可能")

# ---- 2.3 依存関係のインストール（公式 requirements.txt の pin を起点） --------------------------
_pip = [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir"] + PIP_PINNED + EXTRA_PIP_PACKAGES
print("pip install ...", len(PIP_PINNED + EXTRA_PIP_PACKAGES), "packages (数分かかる)")
_r = subprocess.run(_pip, capture_output=True, text=True)
if _r.returncode != 0:
    print(_r.stdout[-3000:], _r.stderr[-3000:])
    raise RuntimeError("pip install failed")
print("pip install done")

# flash-attn: 公式 trainer は attn_implementation='flash_attention_2' 固定。prebuilt wheel を torch/CUDA/Python/ABI から選ぶ
_probe = subprocess.run([sys.executable, "-c",
                         "import torch, json, sys; print(json.dumps(dict(torch=torch.__version__, cuda=torch.version.cuda, "
                         "abi=torch._C._GLIBCXX_USE_CXX11_ABI, py=f'cp{sys.version_info.major}{sys.version_info.minor}', "
                         "cuda_available=torch.cuda.is_available(), "
                         "cap=(torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None))))"],
                        capture_output=True, text=True)
if _probe.returncode != 0:
    print(_probe.stderr[-3000:]); raise RuntimeError("torch probe failed")
TORCH_PROBE = json.loads(_probe.stdout.strip().splitlines()[-1])
print("torch probe:", TORCH_PROBE)
FLASH_ATTN_OK = False
if TORCH_PROBE["cuda_available"]:
    if tuple(TORCH_PROBE["cap"]) < (8, 0):
        raise SystemExit(f"GPU compute capability {TORCH_PROBE['cap']} では flash-attn 2 が動かない（Ampere 以上が必要）。"
                         "公式 trainer は flash_attention_2 固定なので、GPU を A100/L4 などに変更する（コードは変更しない）")
    _tv = ".".join(TORCH_PROBE["torch"].split("+")[0].split(".")[:2])
    _cu = "cu" + TORCH_PROBE["cuda"].split(".")[0]
    _abi = "TRUE" if TORCH_PROBE["abi"] else "FALSE"
    _whl = (f"https://github.com/Dao-AILab/flash-attention/releases/download/v{FLASH_ATTN_VERSION}/"
            f"flash_attn-{FLASH_ATTN_VERSION}+{_cu}torch{_tv}cxx11abi{_abi}-{TORCH_PROBE['py']}-{TORCH_PROBE['py']}-linux_x86_64.whl")
    print("flash-attn wheel:", _whl)
    _r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", _whl], capture_output=True, text=True)
    if _r.returncode != 0:
        print(_r.stderr[-2000:])
        print("prebuilt wheel が無い。ソースビルドを試す（時間がかかる）")
        _r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "--no-build-isolation", f"flash-attn=={FLASH_ATTN_VERSION}"],
                            capture_output=True, text=True)
        if _r.returncode != 0:
            print(_r.stderr[-3000:]); raise RuntimeError("flash-attn のインストールに失敗")
    _r = subprocess.run([sys.executable, "-c", "import flash_attn, flash_attn.bert_padding; print(flash_attn.__version__)"], capture_output=True, text=True)
    FLASH_ATTN_OK = _r.returncode == 0
    print("flash_attn import:", "OK " + _r.stdout.strip() if FLASH_ATTN_OK else "FAILED\n" + _r.stderr[-2000:])
else:
    print("CUDA なし: flash-attn は入れない（公式 trainer は CUDA 無しでは flash_attn を import しない）")

# 公式 trainer モジュールの import 確認（学習と同じ subprocess 環境）
_env = dict(os.environ, PYTHONPATH=REPO_DIR + os.pathsep + os.environ.get("PYTHONPATH", ""))
_chk = subprocess.run([sys.executable, "-c", (
    "import json, importlib; mods=['torch','transformers','accelerate','datasets','tensordict','torchdata','peft','hydra','omegaconf','wandb','ray','codetiming','numpy','pandas','pyarrow','math_verify','huggingface_hub'];\n"
    "vers={m: getattr(importlib.import_module(m), '__version__', 'n/a') for m in mods}\n"
    "import verl.trainer.fsdp_sft_trainer_ours as t\n"
    "import verl.model_merger.fsdp_model_merger\n"
    "print('IMPORT_OK ' + json.dumps(vers))")], capture_output=True, text=True, env=_env, cwd=REPO_DIR)
_ok_lines = [l for l in _chk.stdout.splitlines() if l.startswith("IMPORT_OK")]
if _chk.returncode != 0 or not _ok_lines:
    print(_chk.stdout[-2000:], _chk.stderr[-4000:])
    raise RuntimeError("公式 trainer の import に失敗。不足パッケージを EXTRA_PIP_PACKAGES に追加して再実行")
PKG_VERSIONS = json.loads(_ok_lines[-1][len("IMPORT_OK "):])
import huggingface_hub as _hub_in_kernel
PKG_VERSIONS["huggingface_hub(kernel)"] = _hub_in_kernel.__version__   # カーネル側は cell 0 で import 済みなので実際の版を記録する
if PKG_VERSIONS["huggingface_hub(kernel)"] != PKG_VERSIONS.get("huggingface_hub"):
    print(f"WARN: カーネルの huggingface_hub {PKG_VERSIONS['huggingface_hub(kernel)']} と学習 subprocess の {PKG_VERSIONS.get('huggingface_hub')} が異なる。ランタイム再起動後に cell 0 から実行する")
print("import OK:", PKG_VERSIONS)

# ---- 2.4 環境記録 --------------------------------------------------------------------------
ENV_RECORD = dict(
    session_started_at=SESSION_STARTED_AT, in_colab=IN_COLAB, python=platform.python_version(), platform=platform.platform(),
    gpus=GPUS, n_gpus=N_GPUS, ram_total_gb=round(_mem.get("MemTotal", 0) / 1e6, 1), disk_free_gb=round(_disk.free / 1e9, 1),
    torch_probe=TORCH_PROBE, flash_attn_ok=FLASH_ATTN_OK, flash_attn_version=FLASH_ATTN_VERSION, packages=PKG_VERSIONS,
    fork_repo=FORK_REPO_URL, fork_ref=FORK_REF, fork_commit=FORK_COMMIT, upstream_repo=UPSTREAM_REPO_URL,
    upstream_reference_commit=UPSTREAM_REFERENCE_COMMIT, official_code_unchanged=OFFICIAL_CODE_UNCHANGED, official_diff_stat=OFFICIAL_DIFF_STAT,
)
with open(f"{RECORD_DIR}/env_record.json", "w") as f:
    json.dump(ENV_RECORD, f, indent=2, ensure_ascii=False)
print("saved", f"{RECORD_DIR}/env_record.json")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

## 3. データ取得・AO 抽出・全行監査

`teacher_answer` 列は `response` の `</think>` 以降の最後の `\boxed{...}` を原文のまま用いる。抽出失敗行は `null` のまま残し（削除も `answer` 代用もしない）、ゲート `AO_DATA_READY` で本学習を止める。

In [ ]:
# @title 3. データ取得・AO 抽出・全行監査（元 20,480 行・行順・message・advantage を維持し teacher_answer 列を追加）
import hashlib, json, os, re, collections, time
import pyarrow as pa, pyarrow.parquet as pq, pyarrow.compute as pc
import pandas as pd
import multiprocessing
from huggingface_hub import hf_hub_download

# 既存の boxed 抽出・照合処理を流用する（verl/utils/reward_score/math_verify_ours.py: last_boxed_only_string + math-verify）
from verl.utils.reward_score.math_verify_ours import last_boxed_only_string, compute_score as mv_compute_score


def sha256_of(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


# ---- 3.1 取得（revision 固定、sha256 照合） ----
RAW_PARQUET = hf_hub_download(repo_id=DATASET_REPO, filename=DATASET_FILE, repo_type="dataset", revision=DATASET_REVISION,
                              local_dir=f"{DATA_DIR}/raw")
RAW_SHA256 = sha256_of(RAW_PARQUET)
print("raw parquet:", RAW_PARQUET, "\nsha256:", RAW_SHA256)
assert RAW_SHA256 == DATASET_EXPECTED_SHA256, "Math-CoT-20k.parquet の sha256 が想定と異なる。revision を確認する"

RAW_TABLE = pq.read_table(RAW_PARQUET)
assert RAW_TABLE.num_rows == DATASET_EXPECTED_ROWS, RAW_TABLE.num_rows
for _c in ("message", "response", "answer", "advantage"):
    assert _c in RAW_TABLE.column_names, _c
print("columns:", RAW_TABLE.column_names, "| rows:", RAW_TABLE.num_rows)
_raw_df = RAW_TABLE.select(["response", "answer"]).to_pandas()


# ---- 3.2 抽出（監査用に全 box を列挙し、target は既存 helper の last_boxed_only_string で取る） ----
def find_all_boxed(text):
    """監査用: \\boxed の出現を先頭から列挙する。(位置, 文字列 | 'NOBRACE' | None=閉じ括弧なし)"""
    out, start = [], 0
    while True:
        idx = text.find("\\boxed", start)
        if idx < 0:
            break
        k = idx + len("\\boxed")
        while k < len(text) and text[k] == " ":
            k += 1
        if k < len(text) and text[k] == "{":
            depth, i, right = 0, k, None
            while i < len(text):
                if text[i] == "{":
                    depth += 1
                elif text[i] == "}":
                    depth -= 1
                    if depth == 0:
                        right = i
                        break
                i += 1
            if right is None:
                out.append((idx, None)); start = k + 1
            else:
                out.append((idx, text[idx:right + 1])); start = right + 1
        else:
            out.append((idx, "NOBRACE")); start = k
    return out


def extract_ao_target(response, style):
    """response（教師 CoT+回答）から AO target を抽出する。戻り値: (target or None, status, info)"""
    info = dict(n_think_open=response.count("<think>"), n_think_close=response.count("</think>"))
    after = response.split("</think>")[-1] if info["n_think_close"] >= 1 else response
    boxes_after = find_all_boxed(after)
    info["n_boxes_after_think"] = len(boxes_after)
    info["n_boxes_total"] = len(find_all_boxed(response))
    info["nobrace_form_after_think"] = any(b[1] == "NOBRACE" for b in boxes_after)
    info["unclosed_box_after_think"] = any(b[1] is None for b in boxes_after)
    distinct = [b[1] for b in boxes_after if isinstance(b[1], str) and b[1] != "NOBRACE"]
    info["n_distinct_boxes_after_think"] = len(set(distinct))
    info["first_box_after_think"] = distinct[0] if distinct else None
    last_box = last_boxed_only_string(after)             # 既存 helper（入れ子括弧対応）
    info["last_box_equals_response_last_box"] = (last_box == last_boxed_only_string(response))
    if last_box is None:
        return None, "no_box", info
    inner = last_box[len("\\boxed{"):-1]
    if inner.strip() == "":
        return None, "empty_box", info
    if style == "last_boxed_verbatim":
        target = last_box
    elif style == "boxed_content":
        target = inner
    elif style == "last_boxed_line":
        pos = after.rfind(last_box)
        ls = after.rfind("\n", 0, pos) + 1
        le = after.find("\n", pos + len(last_box))
        target = after[ls: (le if le >= 0 else len(after))].strip()
    else:
        raise ValueError(style)
    return target, "ok", info


def _verify_row(args):
    target, gold = args
    if target is None:
        return "not_extracted"
    try:
        s = mv_compute_score(solution_str=target, ground_truth=gold)   # 既存の照合処理（gold を $..$ で包み math-verify）
    except Exception as e:  # noqa
        return "verify_error:" + type(e).__name__
    return "match" if float(s) == 1.0 else "mismatch"


t0 = time.time()
_targets, _status, _infos = [], [], []
for resp in _raw_df["response"].tolist():
    t, s, i = extract_ao_target(resp, AO_TARGET_STYLE)
    _targets.append(t); _status.append(s); _infos.append(i)
print(f"extraction done in {time.time() - t0:.1f}s")

t0 = time.time()
# セル内で定義した関数を worker で使うため fork を明示（Colab/Linux）。spawn 環境では動かない
with multiprocessing.get_context("fork").Pool(max(1, min(8, os.cpu_count() or 1))) as _pool:
    _verify = _pool.map(_verify_row, list(zip(_targets, _raw_df["answer"].tolist())), chunksize=64)
print(f"math-verify done in {time.time() - t0:.1f}s")

AUDIT = pd.DataFrame(_infos)
AUDIT.insert(0, "row", range(len(AUDIT)))
AUDIT["extraction_status"] = _status
AUDIT["teacher_answer"] = _targets
AUDIT["answer"] = _raw_df["answer"].values
AUDIT["verify_status"] = _verify
AUDIT["teacher_answer_chars"] = AUDIT["teacher_answer"].map(lambda x: len(x) if x else 0)

# ---- 3.3 監査集計とゲート ----
_summary = dict(
    rows=int(len(AUDIT)), target_style=AO_TARGET_STYLE, dataset_repo=DATASET_REPO, dataset_revision=DATASET_REVISION, raw_sha256=RAW_SHA256,
    extraction_status=AUDIT["extraction_status"].value_counts().to_dict(),
    verify_status=AUDIT["verify_status"].value_counts().to_dict(),
    think_tag_counts=dict(open=AUDIT["n_think_open"].value_counts().to_dict(), close=AUDIT["n_think_close"].value_counts().to_dict()),
    n_boxes_after_think=AUDIT["n_boxes_after_think"].value_counts().sort_index().to_dict(),
    n_distinct_boxes_after_think=AUDIT["n_distinct_boxes_after_think"].value_counts().sort_index().to_dict(),
    rows_multi_distinct_boxes=int((AUDIT["n_distinct_boxes_after_think"] > 1).sum()),
    rows_nobrace_form_after_think=int(AUDIT["nobrace_form_after_think"].sum()),
    rows_unclosed_box_after_think=int(AUDIT["unclosed_box_after_think"].sum()),
    rows_last_box_differs_from_response_last_box=int((~AUDIT["last_box_equals_response_last_box"]).sum()),
    teacher_answer_chars=dict(min=int(AUDIT["teacher_answer_chars"].min()), mean=float(AUDIT["teacher_answer_chars"].mean()), max=int(AUDIT["teacher_answer_chars"].max())),
)
_failed = AUDIT[AUDIT["extraction_status"] != "ok"]
_mismatch = AUDIT[AUDIT["verify_status"] != "match"]
_unacknowledged = sorted(set(_mismatch["row"].tolist()) - set(AO_ACKNOWLEDGED_VERIFY_MISMATCH_ROWS))
_stale_ack = sorted(set(AO_ACKNOWLEDGED_VERIFY_MISMATCH_ROWS) - set(_mismatch["row"].tolist()))
_summary.update(rows_extraction_failed=_failed["row"].tolist(), rows_verify_mismatch=_mismatch["row"].tolist(),
                rows_verify_mismatch_unacknowledged=_unacknowledged, acknowledged_rows_not_mismatching=_stale_ack)
print(json.dumps(_summary, indent=1, ensure_ascii=False))

pd.set_option("display.max_colwidth", 120)
if len(_mismatch):
    print("\n--- math-verify 不一致行（教師原文の box と参照正解。代用も削除もしない） ---")
    print(_mismatch[["row", "teacher_answer", "answer", "verify_status", "n_distinct_boxes_after_think"]].to_string(index=False))
if len(_failed):
    print("\n--- 抽出失敗行 ---")
    print(_failed[["row", "extraction_status", "n_boxes_after_think"]].to_string(index=False))
_multi = AUDIT[AUDIT["n_distinct_boxes_after_think"] > 1]
print(f"\n--- 複数の異なる box を持つ行: {len(_multi)}（最後の box が参照正解と一致した行数: {(int((_multi['verify_status'] == 'match').sum()))}）例 ---")
print(_multi[["row", "n_boxes_after_think", "n_distinct_boxes_after_think", "first_box_after_think", "teacher_answer", "answer", "verify_status"]].head(8).to_string(index=False))

AO_DATA_READY = (len(_failed) == 0) and (len(_unacknowledged) == 0)
if not AO_DATA_READY:
    print("\n*** AO データに未解決の抽出/照合問題がある。同一データ条件の本学習へは進まない。"
          "不一致行を確認し、教師の最終回答として保持するなら AO_ACKNOWLEDGED_VERIFY_MISMATCH_ROWS に行番号を追加する ***")
else:
    print("\nAO_DATA_READY = True（抽出失敗 0、未確認の不一致 0）")

# ---- 3.4 派生データの保存（元テーブルに列を追加するだけ。行順・message・advantage はそのまま） ----
AO_TABLE = RAW_TABLE
for _name, _vals, _type in [("teacher_answer", _targets, pa.string()), ("ao_extraction_status", _status, pa.string()),
                            ("ao_verify_status", _verify, pa.string()), ("ao_source_row", list(range(len(_targets))), pa.int64()),
                            ("ao_n_boxes_after_think", AUDIT["n_boxes_after_think"].tolist(), pa.int64())]:
    AO_TABLE = AO_TABLE.append_column(_name, pa.array(_vals, type=_type))
# 抽出失敗行は teacher_answer=null のまま保存する（削除も answer 代用もしない）。ゲートで本学習を止める。
assert AO_TABLE.num_rows == RAW_TABLE.num_rows
assert AO_TABLE.column("message").equals(RAW_TABLE.column("message")) and AO_TABLE.column("advantage").equals(RAW_TABLE.column("advantage"))
AO_PARQUET = f"{DATA_DIR}/Math-AO-20k.parquet"
pq.write_table(AO_TABLE, AO_PARQUET)
AO_SHA256 = sha256_of(AO_PARQUET)
# 読み戻して元列が同一であることを確認
_rt = pq.read_table(AO_PARQUET)
for _c in RAW_TABLE.column_names:
    assert _rt.column(_c).equals(RAW_TABLE.column(_c)), f"column {_c} changed after round trip"
print("saved AO parquet:", AO_PARQUET, "sha256:", AO_SHA256)

AUDIT.to_csv(f"{RECORD_DIR}/ao_audit_rows.csv", index=False)
with open(f"{RECORD_DIR}/ao_audit_summary.json", "w") as f:
    json.dump(dict(_summary, ao_parquet=AO_PARQUET, ao_sha256=AO_SHA256, ao_data_ready=AO_DATA_READY,
                   acknowledged_verify_mismatch_rows=AO_ACKNOWLEDGED_VERIFY_MISMATCH_ROWS,
                   extraction_method="verl.utils.reward_score.math_verify_ours.last_boxed_only_string on text after </think>; "
                                     "verification = math_verify_ours.compute_score(target, answer)"), f, indent=2, ensure_ascii=False)
_flag_rows = sorted(set(_failed["row"]) | set(_mismatch["row"]) | set(_multi["row"].head(200)))
with open(f"{RECORD_DIR}/ao_flagged_rows.jsonl", "w") as f:
    for _r in _flag_rows:
        _rec = {k: (v.item() if hasattr(v, "item") else v) for k, v in AUDIT.iloc[_r].to_dict().items()}
        _rec["response_tail"] = _raw_df["response"].iloc[_r][-400:]
        f.write(json.dumps(_rec, ensure_ascii=False, default=str) + "\n")
print("audit files:", f"{RECORD_DIR}/ao_audit_summary.json", f"{RECORD_DIR}/ao_audit_rows.csv", f"{RECORD_DIR}/ao_flagged_rows.jsonl")
AO_AUDIT_SUMMARY = _summary

## 4. 長さ・target・loss mask・実効設定の確認

In [ ]:
# @title 4-a. 共通ヘルパー: 公式スクリプトの解析・run 仕様の組み立て（公式実装は変更せず既存引数だけを渡す）
import re, json, os, math, pathlib, random, hashlib, datetime
from omegaconf import OmegaConf
from huggingface_hub import snapshot_download, HfApi

HF_API = HfApi() if HF_LOGGED_IN else None
if HF_CKPT_REPO_ID is None and HF_ACCOUNT_NAME:
    HF_CKPT_REPO_ID = f"{HF_ACCOUNT_NAME}/rethink-sft-ao-checkpoints"   # HF の whoami から決める（GitHub 名からは推測しない）
print("HF checkpoint repo:", HF_CKPT_REPO_ID, "(private=%s)" % HF_CKPT_PRIVATE, "| upload during training:", HF_UPLOAD_CHECKPOINTS)


def hf_repo_exists(repo_id, repo_type="model"):
    if HF_API is None or not repo_id:
        return False
    try:
        HF_API.repo_info(repo_id, repo_type=repo_type)
        return True
    except Exception:
        return False


def hf_run_paths(repo_id, run_id):
    """HF 上の runs/<run_id>/ 配下の global_step_* を列挙（完了マーカー ao_upload_verified.json の有無付き）"""
    if not hf_repo_exists(repo_id):
        return {}
    try:
        files = HF_API.list_repo_files(repo_id, repo_type="model")
    except Exception:
        return {}
    out = {}
    for f in files:
        m = re.match(rf"runs/{re.escape(run_id)}/global_step_(\d+)/(.*)$", f)
        if m:
            out.setdefault(int(m.group(1)), set()).add(m.group(2))
    return {s: dict(files=sorted(v), complete=("ao_upload_verified.json" in v)) for s, v in out.items()}


def parse_official_script(path):
    """training_scripts/*.sh から変数と torchrun の hydra 上書き引数を取り出す（$VAR は解決する）"""
    text = pathlib.Path(path).read_text()
    var = {}
    exported = []
    for m in re.finditer(r"^(export\s+)?([A-Za-z_][A-Za-z0-9_]*)=(.*)$", text, re.M):
        is_export, k, v = bool(m.group(1)), m.group(2), m.group(3).strip()
        if v.startswith("$((") or v.startswith("$(") :
            continue
        v = v.strip().strip('"').strip("'")
        var[k] = v
        if is_export:
            exported.append(k)

    def resolve(v):
        for _ in range(3):
            v = re.sub(r"\$\{([A-Za-z_][A-Za-z0-9_]*)\}|\$([A-Za-z_][A-Za-z0-9_]*)",
                       lambda mm: var.get(mm.group(1) or mm.group(2), mm.group(0)), v)
        return v

    var = {k: resolve(v) for k, v in var.items()}
    m = re.search(r"torchrun(.*?)(?:>>|$)", text, re.S)
    assert m, "torchrun ブロックが見つからない"
    tokens = m.group(1).replace("\\\n", " ").split()
    overrides, torchrun_args, module = {}, [], None
    for i, t in enumerate(tokens):
        if t.startswith("--"):
            torchrun_args.append(resolve(t))
        elif t == "-m":
            module = tokens[i + 1]
        elif "=" in t and not t.startswith("-") and t != module:
            k, v = t.split("=", 1)
            overrides[k] = resolve(v.strip('"').strip("'"))
    env = {k: var[k] for k in exported if k not in ("WANDB_API_KEY", "PYTHONPATH")}
    return dict(path=str(path), vars=var, overrides=overrides, torchrun_args=torchrun_args, module=module, env=env)


def hydra_value(v):
    """hydra の CLI 値（文字列）を Python 値へ。OmegaConf で dataset config を組むときに使う"""
    if isinstance(v, (int, float, bool)) or v is None:
        return v
    s = str(v).strip()
    if s in ("null", "None"):
        return None
    if s in ("True", "true"):
        return True
    if s in ("False", "false"):
        return False
    if re.fullmatch(r"-?\d+", s):
        return int(s)
    if re.fullmatch(r"-?\d*\.\d+(e-?\d+)?|-?\d+e-?\d+", s):
        return float(s)
    if s.startswith("[") and s.endswith("]"):
        return json.loads(s.replace("'", '"'))
    return s


def fmt_lr(lr):
    mant, exp = f"{float(lr):e}".split("e")
    mant = mant.rstrip("0").rstrip(".")
    return f"{mant}e{int(exp)}"


OFFICIAL_SCRIPT_PATH = f"{REPO_DIR}/training_scripts/{MODEL_INFO['official_script']}"
OFFICIAL = parse_official_script(OFFICIAL_SCRIPT_PATH)
YAML_DEFAULTS = OmegaConf.load(f"{REPO_DIR}/verl/trainer/config/sft_trainer.yaml")
OFFICIAL_MAX_LENGTH = hydra_value(OFFICIAL["overrides"]["data.max_length"])
OFFICIAL_TBS = hydra_value(OFFICIAL["overrides"]["data.train_batch_size"])
OFFICIAL_MICRO_BSZ = hydra_value(OFFICIAL["overrides"]["data.micro_batch_size_per_gpu"])
OFFICIAL_LR = float(OFFICIAL["overrides"]["optim.lr"])
OFFICIAL_EPOCHS = hydra_value(OFFICIAL["overrides"]["trainer.total_epochs"])

print("公式スクリプト:", OFFICIAL_SCRIPT_PATH)
print("  module:", OFFICIAL["module"], "| torchrun args:", OFFICIAL["torchrun_args"])
print("  env exports:", OFFICIAL["env"])
print("  hydra overrides:")
for _k, _v in OFFICIAL["overrides"].items():
    print(f"    {_k} = {_v}")
print("  yaml 既定（スクリプトで上書きされない主要値）: weight_decay=%s warmup_steps_ratio=%s clip_grad=%s seed=%s strategy=%s grad_ckpt=%s lora_rank=%s"
      % (YAML_DEFAULTS.optim.weight_decay, YAML_DEFAULTS.optim.warmup_steps_ratio, YAML_DEFAULTS.optim.clip_grad, YAML_DEFAULTS.trainer.seed,
         YAML_DEFAULTS.model.strategy, YAML_DEFAULTS.model.enable_gradient_checkpointing, YAML_DEFAULTS.model.lora_rank))
assert hydra_value(YAML_DEFAULTS.model.lora_rank) == 0 and "lora_rank" not in OFFICIAL["overrides"], "Full-parameter SFT のはずが LoRA 設定がある"
assert OFFICIAL_LR == OFFICIAL_BASELINE["lr"] and OFFICIAL_EPOCHS == OFFICIAL_BASELINE["epochs"], "設定セルの OFFICIAL_BASELINE と公式スクリプトが一致しない"


def get_base_model_local():
    """Base モデルを pin した revision で取得し、ローカルパスを返す（model.partial_pretrain に渡す = revision 固定）"""
    d = f"{MODELS_DIR}/{MODEL_KEY}-Base_{MODEL_INFO['base_revision'][:8]}"
    p = snapshot_download(repo_id=MODEL_INFO["base_repo"], revision=MODEL_INFO["base_revision"], local_dir=d)
    return p.rstrip("/")


def steps_per_epoch(n_rows, tbs, n_gpus):
    """DistributedSampler(drop_last) + DataLoader(drop_last) と同じ計算"""
    per_rank_rows = n_rows // n_gpus
    per_rank_bs = tbs // n_gpus
    return per_rank_rows // per_rank_bs


def make_run_id(kind, lr, epochs, tbs=None, suffix=""):
    tbs = tbs or OFFICIAL_TBS
    base = f"{MODEL_KEY}_Math-AO-20k_lr{fmt_lr(lr)}_ep{epochs}_bs{tbs}"
    return f"{base}_{kind}{suffix}"


def build_run_spec(kind, lr, epochs, data_path, data_sha256, n_rows, max_length, save_freq, run_id=None, n_gpus=None,
                   resume_from_path=None, total_training_steps=None, extra_note=None):
    """公式スクリプトの上書き引数を起点に、AO 学習に必要な最小限の引数だけ差し替える。差分は changes に記録する"""
    n_gpus = n_gpus or N_GPUS
    assert n_gpus >= 1, "GPU がない"
    ov = dict(OFFICIAL["overrides"])
    changes = []

    def set_(k, v, reason):
        old = ov.get(k, "<yaml default>")
        v = str(v)
        if old != v:
            changes.append(dict(key=k, official=old, new=v, reason=reason))
        ov[k] = v

    run_id = run_id or make_run_id(kind, lr, epochs, suffix=RUN_ID_SUFFIX)
    base_local = get_base_model_local()
    set_("model.partial_pretrain", base_local, f"同じ Base を revision {MODEL_INFO['base_revision'][:8]} で固定したローカルパス")
    set_("data.train_files", data_path, "AO 派生データ（teacher_answer 列を追加した Math-CoT-20k）")
    set_("data.val_files", data_path, "公式同様 train と同じ値（参照版 trainer では validation はコメントアウト）")
    set_("data.response_key", "teacher_answer", "AO target 列")
    set_("data.max_length", max_length, "AO は短いので全行が収まる長さへ短縮（時間短縮。公式は 20000）" if max_length != OFFICIAL_MAX_LENGTH else "公式値")
    set_("optim.lr", fmt_lr(lr), "探索対象" if float(lr) != OFFICIAL_LR else "公式値")
    set_("trainer.total_epochs", epochs, "探索対象" if int(epochs) != OFFICIAL_EPOCHS else "公式値")
    set_("trainer.save_freq", save_freq, "checkpoint 周期（正整数）")
    set_("trainer.checkpoint.save_contents", '["model","optimizer","extra"]', "再開に必要な状態一式を保存（公式 CoT は model のみ）")
    set_("trainer.checkpoint.load_contents", '["model","optimizer","extra"]', "再開時に一式を読む")
    set_("trainer.default_local_dir", f"{CKPT_DIR}/{run_id}", "run 別ディレクトリ")
    set_("trainer.experiment_name", run_id, "run ID")
    set_("trainer.project_name", WANDB_PROJECT, "wandb project")
    set_("trainer.nnodes", 1, "Colab 単一ノード")
    if MICRO_BATCH_OVERRIDE is not None:
        set_("data.micro_batch_size_per_gpu", MICRO_BATCH_OVERRIDE, "メモリ都合の micro batch 変更（global batch 256 は維持）")
    if CPU_OFFLOAD_OVERRIDE is not None:
        set_("model.fsdp_config.cpu_offload", bool(CPU_OFFLOAD_OVERRIDE), "既存の CPU offload 機能")
        set_("model.fsdp_config.offload_params", bool(CPU_OFFLOAD_OVERRIDE), "既存の CPU offload 機能")
    if resume_from_path:
        set_("trainer.resume_mode", "resume_path", "HF から取得した checkpoint から再開")
        set_("trainer.resume_from_path", resume_from_path, "再開元")
    else:
        set_("trainer.resume_mode", "disable", "新規 run（同じ Base から独立に開始）")
    if total_training_steps is not None:
        set_("trainer.total_training_steps", total_training_steps, "試走専用の step 上限（scheduler 期間も変わる。本学習では使わない）")

    tbs = hydra_value(ov["data.train_batch_size"])
    micro = hydra_value(ov["data.micro_batch_size_per_gpu"])
    assert tbs % n_gpus == 0 and (tbs // n_gpus) % micro == 0, f"train_batch_size {tbs} / n_gpus {n_gpus} / micro {micro} の整合が取れない"
    spe = steps_per_epoch(n_rows, tbs, n_gpus)
    total_steps = int(math.ceil(spe * float(epochs))) if total_training_steps is None else int(total_training_steps)
    warmup = int(total_steps * float(YAML_DEFAULTS.optim.warmup_steps_ratio))
    env = dict(OFFICIAL["env"]) if KEEP_OFFICIAL_ENV_VARS else {k: v for k, v in OFFICIAL["env"].items() if k != "CUDA_LAUNCH_BLOCKING"}
    if not KEEP_OFFICIAL_ENV_VARS:
        changes.append(dict(key="env:CUDA_LAUNCH_BLOCKING", official="1", new="<unset>", reason="KEEP_OFFICIAL_ENV_VARS=False（速度優先。公式は 1）"))
    env["WANDB_MODE"] = WANDB_MODE if (WANDB_MODE == "offline" or os.environ.get("WANDB_API_KEY")) else "offline"
    env["WANDB_DIR"] = f"{WORK_DIR}/wandb"
    env["VERL_SFT_LOGGING_LEVEL"] = "INFO"     # 再開ログ（checkpoint 読み込み等）を表示する。既存の環境変数
    changes.append(dict(key="env:WANDB_DIR / VERL_SFT_LOGGING_LEVEL / PYTHONUNBUFFERED", official="<unset>", new=f"{env['WANDB_DIR']} / INFO / 1",
                        reason="wandb 出力先・再開ログ表示・ログ即時出力（学習内容に影響しない環境変数）"))
    changes.append(dict(key="torchrun --nproc_per_node", official="$PROC_PER_NODE (8 x H200)", new=str(n_gpus),
                        reason=f"実 GPU 台数。global batch {tbs} は維持し、1 GPU あたり {(tbs // n_gpus) // micro} micro batch の勾配蓄積で実現"))
    spec = dict(
        run_id=run_id, kind=kind, lr=float(lr), lr_str=fmt_lr(lr), epochs=epochs, model_key=MODEL_KEY, base_repo=MODEL_INFO["base_repo"],
        base_revision=MODEL_INFO["base_revision"], base_local=base_local, cot_repo=MODEL_INFO["cot_repo"], cot_revision=MODEL_INFO["cot_revision"],
        cot_subfolder=MODEL_INFO["cot_subfolder"], teacher=TEACHER, data_path=data_path, data_sha256=data_sha256, n_rows=n_rows,
        dataset_repo=DATASET_REPO, dataset_revision=DATASET_REVISION, target_style=AO_TARGET_STYLE, max_length=int(max_length),
        official_max_length=OFFICIAL_MAX_LENGTH, tbs=tbs, micro_bsz=micro, n_gpus=n_gpus, grad_accum_micro_batches=(tbs // n_gpus) // micro,
        steps_per_epoch=spe, total_steps=total_steps, warmup_steps=warmup, save_freq=int(save_freq),
        expected_save_steps=sorted(set([s for s in range(int(save_freq), total_steps + 1, int(save_freq))] + [total_steps])),
        overrides=ov, changes_vs_official=changes, env=env, official_script=OFFICIAL["path"], module=OFFICIAL["module"],
        fork_commit=FORK_COMMIT, upstream_reference_commit=UPSTREAM_REFERENCE_COMMIT, keep_official_env_vars=KEEP_OFFICIAL_ENV_VARS,
        resume_from_path=resume_from_path, total_training_steps_override=total_training_steps, note=extra_note,
        created_at=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    )
    return spec


def print_run_spec(spec):
    print(f"run_id={spec['run_id']} kind={spec['kind']} lr={spec['lr_str']} epochs={spec['epochs']} n_gpus={spec['n_gpus']}")
    print(f"  data={spec['data_path']} rows={spec['n_rows']} sha256={spec['data_sha256'][:12]} max_length={spec['max_length']} (official {spec['official_max_length']})")
    print(f"  global batch={spec['tbs']} micro/gpu={spec['micro_bsz']} grad-accum micro-batches/step={spec['grad_accum_micro_batches']}")
    print(f"  steps/epoch={spec['steps_per_epoch']} total_steps={spec['total_steps']} warmup={spec['warmup_steps']} save_freq={spec['save_freq']} -> saves at {spec['expected_save_steps']}")
    print("  公式スクリプトとの差分:")
    for c in spec["changes_vs_official"]:
        print(f"    {c['key']}: {c['official']} -> {c['new']}   [{c['reason']}]")
    if not spec["keep_official_env_vars"]:
        print("    env: CUDA_LAUNCH_BLOCKING を外した（KEEP_OFFICIAL_ENV_VARS=False）")


print("helpers ready: parse_official_script / build_run_spec / print_run_spec / get_base_model_local / fmt_lr")

In [ ]:
# @title 4-b. 長さ・target・loss mask・実効設定の確認（公式 tokenizer / chat template / OurSFTDataset をそのまま使う）
import numpy as np, torch, json, time, math, os, glob, collections
from omegaconf import OmegaConf
from verl.utils import hf_tokenizer
from verl.utils.dataset.sft_dataset import OurSFTDataset

BASE_MODEL_LOCAL = get_base_model_local()
print("base model local:", BASE_MODEL_LOCAL)
TOKENIZER = hf_tokenizer(BASE_MODEL_LOCAL, trust_remote_code=hydra_value(OFFICIAL["overrides"].get("model.trust_remote_code", False)))
# 公式 run_sft と同じ eos 上書き処理（公式スクリプトは data.eos_token=null / eos_token_id=null）
if hydra_value(OFFICIAL["overrides"].get("data.eos_token")) is not None:
    TOKENIZER.eos_token = hydra_value(OFFICIAL["overrides"]["data.eos_token"])
if hydra_value(OFFICIAL["overrides"].get("data.eos_token_id")) is not None:
    TOKENIZER.eos_token_id = hydra_value(OFFICIAL["overrides"]["data.eos_token_id"])
print(f"tokenizer: eos={TOKENIZER.eos_token!r}({TOKENIZER.eos_token_id}) pad={TOKENIZER.pad_token!r}({TOKENIZER.pad_token_id}) bos={TOKENIZER.bos_token!r}")

# ---- 4.1 全行の Q + 回答 + EOS 長（OurSFTDataset.__getitem__ と同じ手順） ----
_msgs = RAW_TABLE.column("message").to_pylist()
_targets_all = AO_TABLE.column("teacher_answer").to_pylist()
t0 = time.time()
PROMPT_LENS = np.array([len(TOKENIZER.apply_chat_template(list(m), add_generation_prompt=True, tokenize=True)) for m in _msgs])
RESP_LENS = np.array([len(TOKENIZER(t if t is not None else "", add_special_tokens=False)["input_ids"]) + 1 for t in _targets_all])  # +1 = EOS
TOTAL_LENS = PROMPT_LENS + RESP_LENS
print(f"tokenized {len(TOTAL_LENS)} rows in {time.time() - t0:.1f}s")
for _name, _arr in (("prompt", PROMPT_LENS), ("target+EOS", RESP_LENS), ("total", TOTAL_LENS)):
    print(f"  {_name:11s} min={_arr.min()} mean={_arr.mean():.1f} p50={int(np.median(_arr))} p95={int(np.percentile(_arr, 95))} p99={int(np.percentile(_arr, 99))} max={_arr.max()}")
_example_prompt = TOKENIZER.apply_chat_template(list(_msgs[0]), add_generation_prompt=True, tokenize=False)
print("chat template 例 (row 0):", repr(_example_prompt[:120]), "...", repr(_example_prompt[-40:]))
print("target 例 (row 0):", repr(_targets_all[0]), "-> tokens", TOKENIZER.convert_ids_to_tokens(TOKENIZER(_targets_all[0], add_special_tokens=False)["input_ids"]), "+ EOS")

MAX_TOTAL_LEN = int(TOTAL_LENS.max())
AUTO_FIT_MAX_LENGTH = int(math.ceil(MAX_TOTAL_LEN / MAX_LENGTH_ROUND_TO) * MAX_LENGTH_ROUND_TO)
if MAX_LENGTH_MODE == "auto_fit":
    MAX_LENGTH = AUTO_FIT_MAX_LENGTH
elif MAX_LENGTH_MODE == "official":
    MAX_LENGTH = OFFICIAL_MAX_LENGTH
else:
    MAX_LENGTH = int(MAX_LENGTH_MODE)
_n_trunc = int((TOTAL_LENS > MAX_LENGTH).sum())
print(f"\n全行が切り詰められずに収まる最大長 = {MAX_TOTAL_LEN} tokens（{MAX_LENGTH_ROUND_TO} の倍数へ切り上げ: {AUTO_FIT_MAX_LENGTH}）")
print(f"data.max_length = {MAX_LENGTH}（mode={MAX_LENGTH_MODE}、公式 {OFFICIAL_MAX_LENGTH}）。この長さで切り詰められる行数: {_n_trunc}")
print(f"公式 20000 での padding 率: {(1 - TOTAL_LENS.mean() / OFFICIAL_MAX_LENGTH) * 100:.1f}% / 選択値での padding 率: {(1 - TOTAL_LENS.mean() / MAX_LENGTH) * 100:.1f}%")
if _n_trunc > 0:
    print("*** 切り詰められる行がある。data.truncation=right で target/EOS が失われる → max_length を上げる ***")

# ---- 4.2 公式 OurSFTDataset で loss mask を確認 ----
_data_cfg = OmegaConf.create(OmegaConf.to_container(YAML_DEFAULTS.data, resolve=True))
for _k, _v in OFFICIAL["overrides"].items():
    if _k.startswith("data."):
        OmegaConf.update(_data_cfg, _k[len("data."):], hydra_value(_v), merge=True)
OmegaConf.update(_data_cfg, "train_files", AO_PARQUET)
OmegaConf.update(_data_cfg, "val_files", AO_PARQUET)
OmegaConf.update(_data_cfg, "response_key", "teacher_answer")
OmegaConf.update(_data_cfg, "max_length", int(MAX_LENGTH))
print("\ndataset config (data.*):", json.dumps(OmegaConf.to_container(_data_cfg), ensure_ascii=False)[:600])
DS = OurSFTDataset(parquet_files=AO_PARQUET, tokenizer=TOKENIZER, config=_data_cfg)
assert len(DS) == DATASET_EXPECTED_ROWS


def check_item(i):
    it = DS[i]
    p, r = int(PROMPT_LENS[i]), int(RESP_LENS[i])
    ids, am, lm = it["input_ids"], it["attention_mask"], it["loss_mask"]
    ok = dict(
        length=(ids.shape[0] == MAX_LENGTH),
        no_truncation=(int(am.sum()) == p + r),
        loss_mask_count=(int(lm.sum()) == r),                                              # target tokens + EOS の数
        loss_mask_positions=(torch.nonzero(lm).flatten().tolist() == list(range(p - 1, p + r - 1))),  # 1 つ前の位置が次 token を予測
        prompt_masked=(int(lm[: p - 1].sum()) == 0),
        padding_masked=(int(lm[p + r - 1:].sum()) == 0 and int(am[p + r:].sum()) == 0),
        eos_supervised=(int(ids[p + r - 1]) == TOKENIZER.eos_token_id and int(lm[p + r - 2]) == 1),
        target_tokens=(ids[p: p + r - 1].tolist() == TOKENIZER(_targets_all[i], add_special_tokens=False)["input_ids"]),
        advantage=(float(it["advantages"][0]) == 1.0),
    )
    return ok, it, p, r


t0 = time.time()
_bad = collections.Counter()
FULL_MASK_SCAN = True
_null_rows = [i for i, t in enumerate(_targets_all) if t is None]   # 抽出失敗行（OurSFTDataset は str を要求するので走査から除く。ゲートは AO_DATA_READY）
_scan_rows = [i for i in (range(len(DS)) if FULL_MASK_SCAN else range(0, len(DS), 97)) if _targets_all[i] is not None]
for _i in _scan_rows:
    _ok, _, _, _ = check_item(_i)
    for _k, _v in _ok.items():
        if not _v:
            _bad[_k] += 1
print(f"\nloss mask scan over {len(_scan_rows)} rows in {time.time() - t0:.1f}s -> failures: {dict(_bad) or 'none'} | skipped null-target rows: {len(_null_rows)}")
MASK_CHECK_OK = (len(_bad) == 0) and (len(_null_rows) == 0)
for _i in [i for i in (0, int(TOTAL_LENS.argmax())) if _targets_all[i] is not None]:
    _ok, _it, _p, _r = check_item(_i)
    _sup = _it["input_ids"][_p: _p + _r]
    print(f"row {_i}: prompt_len={_p} target+eos={_r} supervised tokens -> {TOKENIZER.decode(_sup)!r}  checks={_ok}")

# ---- 4.3 実効設定 ----
_param_bytes = sum(os.path.getsize(f) for f in glob.glob(f"{BASE_MODEL_LOCAL}/*.safetensors"))
CKPT_SIZE_EST_GB = _param_bytes * 3 / 1e9   # bf16 params + AdamW の 2 状態（公式は model_dtype=bf16 で読み込む）
print("\n=== 実効設定（1 GPU あたり micro batch、勾配蓄積） ===")
for _lr, _ep, _kind in [(OFFICIAL_BASELINE["lr"], OFFICIAL_BASELINE["epochs"], "baseline")] + [(a, b, "search") for a, b in SEARCH_RUN_LIST]:
    _spe = steps_per_epoch(DATASET_EXPECTED_ROWS, OFFICIAL_TBS, max(N_GPUS, 1))
    _micro = MICRO_BATCH_OVERRIDE or OFFICIAL_MICRO_BSZ
    _total = int(math.ceil(_spe * _ep))
    _nsave = len(set(list(range(SAVE_FREQ, _total + 1, SAVE_FREQ)) + [_total]))
    print(f"  {_kind:8s} lr={fmt_lr(_lr)} ep={_ep}: steps/epoch={_spe} total={_total} warmup={int(_total * float(YAML_DEFAULTS.optim.warmup_steps_ratio))} "
          f"micro/gpu={_micro} accum={(OFFICIAL_TBS // max(N_GPUS, 1)) // _micro} saves={_nsave} (~{_nsave * CKPT_SIZE_EST_GB:.0f} GB on HF at {CKPT_SIZE_EST_GB:.1f} GB/ckpt)")
print("  loss: token-mean を micro batch ごとに計算し、micro batch 数で平均して backward（公式 training_step）。micro batch を変えると重み付けが僅かに変わる。")
print("  optimizer: AdamW betas=%s wd=%s, scheduler=%s warmup_ratio=%s, clip_grad=%s, bf16, FSDP(%s), grad ckpt=%s, shuffle=%s, truncation=%s"
      % (OFFICIAL["overrides"].get("optim.betas"), YAML_DEFAULTS.optim.weight_decay, OFFICIAL["overrides"].get("optim.lr_scheduler"),
         YAML_DEFAULTS.optim.warmup_steps_ratio, YAML_DEFAULTS.optim.clip_grad, YAML_DEFAULTS.model.strategy, YAML_DEFAULTS.model.enable_gradient_checkpointing,
         OFFICIAL["overrides"].get("data.shuffle_train"), OFFICIAL["overrides"].get("data.truncation")))
LENGTH_RECORD = dict(max_total_len=MAX_TOTAL_LEN, auto_fit_max_length=AUTO_FIT_MAX_LENGTH, chosen_max_length=int(MAX_LENGTH), official_max_length=OFFICIAL_MAX_LENGTH,
                     rows_truncated_at_chosen=_n_trunc, prompt=dict(min=int(PROMPT_LENS.min()), mean=float(PROMPT_LENS.mean()), max=int(PROMPT_LENS.max())),
                     target_eos=dict(min=int(RESP_LENS.min()), mean=float(RESP_LENS.mean()), max=int(RESP_LENS.max())), mask_check_ok=MASK_CHECK_OK,
                     mask_scan_rows=len(_scan_rows), mask_failures=dict(_bad), ckpt_size_est_gb=CKPT_SIZE_EST_GB, tokenizer_eos=TOKENIZER.eos_token, tokenizer_pad=TOKENIZER.pad_token)
with open(f"{RECORD_DIR}/length_and_mask_record.json", "w") as f:
    json.dump(LENGTH_RECORD, f, indent=2, ensure_ascii=False)
del DS

## 5. ドライラン（既定の実行モード）

In [ ]:
# @title 5. ドライラン: データ・設定・コマンド・想定 step 数・保存先を確認する（学習も HF 書き込みも開始しない）
import shutil, json, os


def plan_runs():
    """RUN_KIND に応じて実行予定の (lr, epochs) を返す"""
    if RUN_KIND == "baseline":
        return [(OFFICIAL_BASELINE["lr"], OFFICIAL_BASELINE["epochs"])]
    return list(SEARCH_RUN_LIST)


DRY_RUN_SPECS = []
if not plan_runs():
    print("実行予定の候補が無い（RUN_KIND='search' なら SEARCH_RUN_LIST を設定する）")
for _lr, _ep in plan_runs():
    _spec = build_run_spec(RUN_KIND, _lr, _ep, data_path=AO_PARQUET, data_sha256=AO_SHA256, n_rows=DATASET_EXPECTED_ROWS,
                           max_length=MAX_LENGTH, save_freq=SAVE_FREQ, n_gpus=max(N_GPUS, 1))
    DRY_RUN_SPECS.append(_spec)
    print("=" * 100)
    print_run_spec(_spec)
    _cmd = ["torchrun", "--nnodes=1", f"--nproc_per_node={_spec['n_gpus']}", "--node_rank=0", "--master_addr=127.0.0.1", "--master_port=<random>",
            "-m", _spec["module"]] + [f"{k}={v}" for k, v in _spec["overrides"].items()]
    print("  command:\n    " + " \\\n    ".join(_cmd))
    print("  env:", _spec["env"])
    print(f"  checkpoint 容量見積: {len(_spec['expected_save_steps'])} saves x {CKPT_SIZE_EST_GB:.1f} GB = {len(_spec['expected_save_steps']) * CKPT_SIZE_EST_GB:.0f} GB（HF）、ローカルは検証済み {LOCAL_KEEP_LAST_N_VERIFIED_CKPTS} 個を保持")
    _local_exists = os.path.exists(_spec["overrides"]["trainer.default_local_dir"])
    _remote = hf_run_paths(HF_CKPT_REPO_ID, _spec["run_id"]) if HF_API else {}
    print(f"  衝突確認: local dir exists={_local_exists} | HF runs/{_spec['run_id']}: {sorted(_remote) if _remote else 'なし'}")
    if _local_exists or _remote:
        print("  *** 同名 run が既に存在する。上書きしない。再開なら『再開』セル、別 run なら RUN_ID_SUFFIX を設定 ***")
    with open(f"{RECORD_DIR}/dry_run_{_spec['run_id']}.json", "w") as f:
        json.dump(dict(spec=_spec, command=_cmd, local_exists=_local_exists, remote_steps=_remote), f, indent=2, ensure_ascii=False)

print("=" * 100)
print("torchrun:", shutil.which("torchrun"), "| GPUs:", N_GPUS, "| AO_DATA_READY:", AO_DATA_READY, "| MASK_CHECK_OK:", MASK_CHECK_OK, "| flash_attn:", FLASH_ATTN_OK)
print("ドライラン完了。学習・HF 書き込みは行っていない。RUN_MODE を 'trial' / 'train' にして先へ進む。")

## 6. 短い学習・HF 保存・再開の試走

6-a は起動・監視・HF 転送・再開の共通処理（セクション 7〜10 と『再開』でも使う）。6-b が試走本体。

In [ ]:
# @title 6-a. 起動・監視・HF 転送・再開のヘルパー（追加処理はすべてここ。公式 trainer は subprocess で無変更のまま実行）
import subprocess, threading, time, json, os, re, glob, shutil, zipfile, hashlib, random, datetime, io, sys, signal
import numpy as np
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

HF_API = HfApi() if HF_LOGGED_IN else None


def now_iso():
    return datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")


def hash_file(path, with_sha256=True):
    """1 回の読み込みで git blob sha1（小さいファイルの照合用）と sha256（LFS 照合用）を計算する"""
    size = os.path.getsize(path)
    h1 = hashlib.sha1(f"blob {size}\0".encode())
    h2 = hashlib.sha256() if with_sha256 else None
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 22), b""):
            h1.update(b)
            if h2 is not None:
                h2.update(b)
    return h1.hexdigest(), (h2.hexdigest() if h2 is not None else None)


def git_blob_sha1(path):
    return hash_file(path, with_sha256=False)[0]


# ---- checkpoint の完全性 -----------------------------------------------------------------------
def checkpoint_expected_files(world_size):
    files = []
    for r in range(world_size):
        files += [f"model_world_size_{world_size}_rank_{r}.pt", f"optim_world_size_{world_size}_rank_{r}.pt", f"extra_state_world_size_{world_size}_rank_{r}.pt"]
    files += ["fsdp_config.json", "data.pt", "huggingface/config.json", "huggingface/tokenizer_config.json"]
    return files


def checkpoint_is_complete(ckpt_dir, world_size, prev_sizes=None, require_stable=True):
    """必要ファイルの存在・非空・torch.save(zip) の完全性・tracker・（連続 2 回のスキャンで）サイズ不変 を確認する。
    ディレクトリの存在や latest tracker だけでは完了と判断しない。"""
    step = int(re.search(r"global_step_(\d+)", ckpt_dir).group(1))
    sizes = {}
    for rel in checkpoint_expected_files(world_size):
        p = os.path.join(ckpt_dir, rel)
        if not os.path.isfile(p) or os.path.getsize(p) == 0:
            return False, sizes
        sizes[rel] = os.path.getsize(p)
        if rel.endswith(".pt") and not zipfile.is_zipfile(p):   # torch.save は zip 形式。末尾まで書けていれば central directory がある
            return False, sizes
    tokenizer_ok = any(os.path.isfile(os.path.join(ckpt_dir, "huggingface", f)) for f in ("tokenizer.json", "vocab.json", "tokenizer.model"))
    if not tokenizer_ok:
        return False, sizes
    tracker = os.path.join(os.path.dirname(ckpt_dir), "latest_checkpointed_iteration.txt")
    try:
        if int(open(tracker).read().strip()) < step:
            return False, sizes
    except Exception:
        return False, sizes
    for root, _, fs in os.walk(ckpt_dir):
        for f in fs:
            rel = os.path.relpath(os.path.join(root, f), ckpt_dir)
            sizes[rel] = os.path.getsize(os.path.join(root, f))
    if require_stable and prev_sizes != sizes:
        return False, sizes
    return True, sizes


def build_manifest(spec, step, ckpt_dir, with_sha256=None):
    with_sha256 = CKPT_MANIFEST_SHA256 if with_sha256 is None else with_sha256
    files = []
    for root, _, fs in os.walk(ckpt_dir):
        for f in sorted(fs):
            p = os.path.join(root, f)
            rel = os.path.relpath(p, ckpt_dir)
            if rel in ("ao_ckpt_manifest.json", "ao_upload_verified.json"):
                continue
            sha1, sha256 = hash_file(p, with_sha256=with_sha256)
            ent = dict(path=rel, size=os.path.getsize(p), git_blob_sha1=sha1)
            if sha256:
                ent["sha256"] = sha256
            files.append(ent)
    manifest = dict(run_id=spec["run_id"], step=step, world_size=spec["n_gpus"], files=files, total_bytes=sum(f["size"] for f in files),
                    run_config=spec, created_at=now_iso(), fork_commit=FORK_COMMIT, data_sha256=spec["data_sha256"], base_revision=spec["base_revision"])
    with open(os.path.join(ckpt_dir, "ao_ckpt_manifest.json"), "w") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)
    return manifest


# ---- HF 転送 -----------------------------------------------------------------------------------
def ensure_ckpt_repo():
    assert HF_API is not None, "HF にログインしていない（最初のセルで HF_TOKEN を設定）"
    assert HF_CKPT_REPO_ID, "HF_CKPT_REPO_ID が未設定"
    HF_API.create_repo(HF_CKPT_REPO_ID, private=HF_CKPT_PRIVATE, repo_type="model", exist_ok=True)


def hf_verify_uploaded(repo_id, path_in_repo, manifest, revision):
    """転送後に HF 側のサイズ・LFS sha256・git blob sha1 を manifest と照合する"""
    paths = [f"{path_in_repo}/{e['path']}" for e in manifest["files"]] + [f"{path_in_repo}/ao_ckpt_manifest.json"]
    infos = {i.path: i for i in HF_API.get_paths_info(repo_id, paths=paths, revision=revision, repo_type="model")}
    mismatches = []
    for e in manifest["files"]:
        rp = f"{path_in_repo}/{e['path']}"
        i = infos.get(rp)
        if i is None:
            mismatches.append((e["path"], "missing on HF")); continue
        if getattr(i, "size", None) != e["size"]:
            mismatches.append((e["path"], f"size {getattr(i, 'size', None)} != {e['size']}")); continue
        lfs = getattr(i, "lfs", None)
        if lfs is not None and e.get("sha256") and getattr(lfs, "sha256", None) and lfs.sha256 != e["sha256"]:
            mismatches.append((e["path"], "sha256 mismatch"))
        elif lfs is None and getattr(i, "blob_id", None) and i.blob_id != e["git_blob_sha1"]:
            mismatches.append((e["path"], "git blob sha1 mismatch"))
    if f"{path_in_repo}/ao_ckpt_manifest.json" not in infos:
        mismatches.append(("ao_ckpt_manifest.json", "missing on HF"))
    return mismatches


def upload_checkpoint(spec, step, ckpt_dir, log=print):
    """保存完了した checkpoint を runs/<run_id>/global_step_<step>/ へ転送し、検証後に完了マーカーを置く。既存の完了済み step は上書きしない"""
    t0 = time.time()
    ensure_ckpt_repo()
    pir = f"runs/{spec['run_id']}/global_step_{step}"
    remote = hf_run_paths(HF_CKPT_REPO_ID, spec["run_id"]).get(step)
    if remote and remote["complete"]:
        log(f"collision: {pir} は HF に完了済みで存在する。上書きしない")
        return dict(status="collision_remote_complete", path_in_repo=pir)
    if remote:
        log(f"{pir} に未完了の転送が残っている。同じ step を再転送する")
    manifest = build_manifest(spec, step, ckpt_dir)
    last_err = None
    for attempt in range(5):
        try:
            info = HF_API.upload_folder(folder_path=ckpt_dir, path_in_repo=pir, repo_id=HF_CKPT_REPO_ID, repo_type="model",
                                        commit_message=f"{spec['run_id']} global_step_{step}", ignore_patterns=["ao_upload_verified.json"])
            break
        except Exception as e:  # noqa
            last_err = e
            wait = 2 ** attempt * 5
            log(f"upload attempt {attempt + 1} failed: {type(e).__name__}: {str(e)[:200]} -> retry in {wait}s")
            time.sleep(wait)
    else:
        return dict(status="upload_failed", error=repr(last_err)[:500], path_in_repo=pir, duration_sec=time.time() - t0)
    mism = hf_verify_uploaded(HF_CKPT_REPO_ID, pir, manifest, info.oid)
    if mism:
        log(f"verify failed for {pir}: {mism[:5]}")
        return dict(status="verify_failed", commit=info.oid, mismatches=mism, path_in_repo=pir, duration_sec=time.time() - t0)
    marker = dict(run_id=spec["run_id"], step=step, folder_commit=info.oid, verified_at=now_iso(), n_files=len(manifest["files"]),
                  total_bytes=manifest["total_bytes"], world_size=spec["n_gpus"], total_steps=spec["total_steps"])
    with open(os.path.join(ckpt_dir, "ao_upload_verified.json"), "w") as f:
        json.dump(marker, f, indent=2)
    info2 = HF_API.upload_file(path_or_fileobj=os.path.join(ckpt_dir, "ao_upload_verified.json"), path_in_repo=f"{pir}/ao_upload_verified.json",
                               repo_id=HF_CKPT_REPO_ID, repo_type="model", commit_message=f"{spec['run_id']} global_step_{step} verified")
    dur = time.time() - t0
    log(f"uploaded+verified {pir} ({manifest['total_bytes'] / 1e9:.2f} GB, {dur:.0f}s) commit={info.oid[:10]} marker={info2.oid[:10]}")
    return dict(status="verified", commit=info.oid, marker_commit=info2.oid, path_in_repo=pir, total_bytes=manifest["total_bytes"], duration_sec=dur,
                url=f"https://huggingface.co/{HF_CKPT_REPO_ID}/tree/{info2.oid}/{pir}")


# ---- 監視スレッド ------------------------------------------------------------------------------
class TrainingMonitor(threading.Thread):
    """学習プロセスと並行して checkpoint ディレクトリを監視し、保存完了ごとに HF へ転送・検証する。GPU メモリも記録する。
    kill_after_step が指定された場合（試走）: その step の保存完了を検知したら学習プロセス群を SIGSTOP で一時停止し、
    HF 転送・検証が終わってから SIGTERM で終了する（保存 → HF 転送 → プロセス終了 の順を保証する）。"""

    def __init__(self, proc, spec, kill_after_step=None, upload=True, poll_sec=10, keep_local=None, stable_scans=True):
        super().__init__(daemon=True)
        self.proc, self.spec, self.kill_after_step, self.upload, self.poll_sec = proc, spec, kill_after_step, upload, poll_sec
        self.keep_local = LOCAL_KEEP_LAST_N_VERIFIED_CKPTS if keep_local is None else keep_local
        self.stable_scans = stable_scans   # True: 連続 2 回のスキャンでサイズ不変を要求（通常）。試走では tracker + zip 検証のみで即時判定
        self.ckpt_root = spec["overrides"]["trainer.default_local_dir"]
        self.state = dict(uploaded={}, failed={}, complete_local=[], gpu_mem_max_mb={}, events=[], deleted_local=[])
        self._sizes, self._attempts, self.killed, self.paused = {}, {}, False, False

    def _signal_group(self, sig):
        try:
            os.killpg(os.getpgid(self.proc.pid), sig)
            return True
        except ProcessLookupError:
            return False

    def _terminate(self):
        alive = self.proc.poll() is None
        if not alive:
            self.log("プロセスは既に終了していた（kill は不要）")
            return False
        if self.paused:
            self._signal_group(signal.SIGCONT)
            self.paused = False
        self._signal_group(signal.SIGTERM)
        try:
            self.proc.wait(timeout=60)
        except subprocess.TimeoutExpired:
            self._signal_group(signal.SIGKILL)
        self.killed = True
        return True

    def log(self, msg):
        line = f"[monitor {time.strftime('%H:%M:%S')}] {msg}"
        print(line, flush=True)
        self.state["events"].append(dict(t=time.time(), msg=msg))

    def _sample_gpu(self):
        if shutil.which("nvidia-smi") is None:
            return
        r = subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used", "--format=csv,noheader,nounits"], capture_output=True, text=True)
        for line in r.stdout.splitlines():
            try:
                i, used = [x.strip() for x in line.split(",")]
                self.state["gpu_mem_max_mb"][i] = max(self.state["gpu_mem_max_mb"].get(i, 0), int(float(used)))
            except ValueError:
                pass

    def _cleanup_local(self):
        verified = sorted(int(s) for s in self.state["uploaded"] if self.state["uploaded"][s]["status"] == "verified")
        resume_src = self.spec.get("resume_from_path")
        for s in verified[: max(0, len(verified) - self.keep_local)]:
            d = os.path.join(self.ckpt_root, f"global_step_{s}")
            if resume_src and os.path.abspath(d) == os.path.abspath(resume_src):
                continue   # 再開元は消さない
            if os.path.isdir(d):
                shutil.rmtree(d, ignore_errors=True)
                self.state["deleted_local"].append(s)
                self.log(f"local cleanup: removed global_step_{s} (HF 転送・検証済み)")

    def _scan(self, final=False):
        dirs = glob.glob(os.path.join(self.ckpt_root, "global_step_*"))
        steps = sorted(int(re.search(r"global_step_(\d+)$", d).group(1)) for d in dirs if re.search(r"global_step_(\d+)$", d))
        for step in steps:
            if step in self.state["uploaded"]:
                continue
            d = os.path.join(self.ckpt_root, f"global_step_{step}")
            marker = os.path.join(d, "ao_upload_verified.json")
            if os.path.isfile(marker):   # 以前の実行で転送・検証済み（再開のために HF から取得したもの）。再転送しない
                try:
                    self.state["uploaded"][step] = dict(status="verified", previously_verified=True, **json.load(open(marker)))
                    self.log(f"global_step_{step} は転送・検証済み（マーカーあり）。スキップ")
                except Exception:
                    pass
                continue
            complete, sizes = checkpoint_is_complete(d, self.spec["n_gpus"], self._sizes.get(step), require_stable=(self.stable_scans and not final))
            self._sizes[step] = sizes
            if not complete:
                continue
            if step not in self.state["complete_local"]:
                self.state["complete_local"].append(step)
                self.log(f"checkpoint global_step_{step} complete locally ({sum(sizes.values()) / 1e9:.2f} GB, {len(sizes)} files)")
            kill_now = self.kill_after_step is not None and step >= self.kill_after_step and not self.killed
            if kill_now and not self.paused and self.proc.poll() is None:
                self.paused = self._signal_group(signal.SIGSTOP)   # 転送中に学習が先へ進まないよう一時停止（切断前の状態を固定）
                self.log(f"kill_after_step={self.kill_after_step}: global_step_{step} の保存完了を検知。学習プロセスを一時停止して HF 転送する")
            if self.upload:
                if self._attempts.get(step, 0) >= 3:
                    continue
                self._attempts[step] = self._attempts.get(step, 0) + 1
                res = upload_checkpoint(self.spec, step, d, log=self.log)
                if res["status"] == "verified":
                    self.state["uploaded"][step] = res
                    self.state["failed"].pop(step, None)
                    self._cleanup_local()
                elif res["status"].startswith("collision"):
                    # 同じ step が HF に完了済み（例: 再開元）。上書きしない。転送済みとして扱う
                    self.state["uploaded"][step] = dict(res, status="verified", previously_verified=True)
                    self._attempts[step] = 99
                else:
                    self.state["failed"][step] = res
            if kill_now and (not self.upload or step in self.state["uploaded"]):
                self.log(f"global_step_{step} の{'HF 転送・検証' if self.upload else '保存'}を確認したのでプロセスを終了する（切断の模擬）")
                self._terminate()

    def run(self):
        while True:
            alive = self.proc.poll() is None
            self._sample_gpu()
            try:
                self._scan(final=not alive)
            except Exception as e:  # noqa
                self.log(f"monitor error: {type(e).__name__}: {str(e)[:300]}")
                if self.paused:   # 転送に失敗しても停止したまま放置しない
                    self._terminate()
            if not alive:
                try:
                    self._scan(final=True)
                except Exception as e:  # noqa
                    self.log(f"monitor error: {type(e).__name__}: {str(e)[:300]}")
                break
            time.sleep(self.poll_sec)


# ---- 起動 --------------------------------------------------------------------------------------
_METRIC_RE = re.compile(r"(?:^|\s)step:(\d+)((?: - [\w/]+:\S+)+)")


def parse_metric_line(line):
    """console logger の `step:N - train/loss:0.5 - train/lr:1e-05 ...` を辞書にする（nan/inf も float として保持）"""
    m = _METRIC_RE.search(line)
    if not m:
        return None
    d = dict(step=int(m.group(1)))
    for kv in m.group(2).split(" - ")[1:]:
        k, v = kv.split(":", 1)
        try:
            d[k] = float(v)
        except ValueError:
            pass
    return d


_INTERESTING = ("Saving checkpoint", "Updated checkpoint tracker", "Loaded", "loaded", "resume", "Traceback", "Error", "error", "OutOfMemory", "Total training steps",
                "Number of steps/epoch", "Normalize batch", "Using FSDP rank", "WARN", "Killed", "Signal", "Found checkpoint", "grad_norm is not finite")


def launch_training(spec, kill_after_step=None, upload=None, keep_local=None, quiet=False, poll_sec=10, stable_scans=True):
    """公式エントリーポイントを torchrun (python -m torch.distributed.run) で起動し、ログを流しつつ監視スレッドで checkpoint を HF へ転送する。終了まで戻らない。"""
    upload = HF_UPLOAD_CHECKPOINTS if upload is None else upload
    if upload:
        ensure_ckpt_repo()
    run_id = spec["run_id"]
    run_log_dir = f"{LOG_DIR}/{run_id}"
    os.makedirs(run_log_dir, exist_ok=True)
    ts = time.strftime("%Y%m%d-%H%M%S")
    log_path = f"{run_log_dir}/{ts}.log"
    with open(f"{run_log_dir}/{ts}.spec.json", "w") as f:
        json.dump(spec, f, indent=2, ensure_ascii=False)
    port = random.randint(20000, 21000)
    cmd = [sys.executable, "-m", "torch.distributed.run", "--nnodes=1", f"--nproc_per_node={spec['n_gpus']}", "--node_rank=0", "--master_addr=127.0.0.1",
           f"--master_port={port}", "-m", spec["module"]] + [f"{k}={v}" for k, v in spec["overrides"].items()]
    env = os.environ.copy()
    env.update(spec["env"])
    env["PYTHONPATH"] = REPO_DIR + os.pathsep + env.get("PYTHONPATH", "")
    env["PYTHONUNBUFFERED"] = "1"
    with open(f"{run_log_dir}/{ts}.cmd.txt", "w") as f:
        f.write(" \\\n  ".join(cmd) + "\n\nenv: " + json.dumps(spec["env"]) + "\n")
    print(f"launch {run_id}: log={log_path}")
    print("  " + " ".join(cmd[:10]) + " ...")
    t_launch = time.time()
    # start_new_session=True: torchrun とワーカーを 1 つのプロセスグループにし、試走の一時停止/終了をグループごと送れるようにする
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, errors="replace",
                            start_new_session=True)
    mon = TrainingMonitor(proc, spec, kill_after_step=kill_after_step, upload=upload, keep_local=keep_local, poll_sec=poll_sec, stable_scans=stable_scans)
    mon.start()
    metrics, events = [], []
    try:
        with open(log_path, "w") as lf:
            for line in proc.stdout:
                t = time.time()
                lf.write(f"[{t - t_launch:9.1f}s] {line}")
                m = parse_metric_line(line)
                if m:
                    m["t"] = t
                    metrics.append(m)
                    if not quiet:
                        print(f"[{t - t_launch:7.0f}s] step {m['step']}/{spec['total_steps']} loss={m.get('train/loss', float('nan')):.4f} "
                              f"lr={m.get('train/lr', float('nan')):.3e} grad_norm={m.get('train/grad_norm', float('nan')):.3f} entropy={m.get('train/entropy/mean', float('nan')):.3f}", flush=True)
                elif any(k in line for k in _INTERESTING) and "ratio_list_all" not in line:
                    events.append(dict(t=t, line=line.rstrip()[:300]))
                    if not quiet:
                        print(f"[{t - t_launch:7.0f}s] {line.rstrip()[:200]}", flush=True)
    except KeyboardInterrupt:
        print("KeyboardInterrupt -> terminating training process group")
        mon._terminate()
    rc = proc.wait()
    try:
        mon.join(timeout=3600 * 6)   # プロセス終了後も、最後の checkpoint の転送が終わるまで待つ
    except KeyboardInterrupt:
        print("KeyboardInterrupt during final upload -> 結果は転送未完了のまま記録する")
    t_end = time.time()
    result = dict(run_id=run_id, spec=spec, exit_code=rc, killed_by_monitor=mon.killed, log_path=log_path, launched_at=t_launch, ended_at=t_end,
                  wall_sec=t_end - t_launch, metrics=metrics, events=events, monitor=dict(uploaded=mon.state["uploaded"], failed=mon.state["failed"],
                  complete_local=mon.state["complete_local"], gpu_mem_max_mb=mon.state["gpu_mem_max_mb"], deleted_local=mon.state["deleted_local"],
                  events=mon.state["events"]))
    with open(f"{run_log_dir}/{ts}.result.json", "w") as f:
        json.dump(result, f, indent=2, ensure_ascii=False, default=str)
    status = "killed(simulated disconnect)" if mon.killed else ("OK" if rc == 0 else f"FAILED rc={rc}")
    print(f"finished {run_id}: {status} | steps logged={len(metrics)} | wall={t_end - t_launch:.0f}s | GPU mem max MB={mon.state['gpu_mem_max_mb']} "
          f"| uploaded={sorted(mon.state['uploaded'])} | failed={ {k: v['status'] for k, v in mon.state['failed'].items()} }")
    if mon.state["failed"]:
        print("*** HF 転送に失敗した checkpoint がある。ローカルには残している。ネットワーク復旧後に upload_checkpoint() を再実行する ***")
    return result


def summarize_timing(result):
    """初期化時間・optimizer step 時間（勾配蓄積込み）・保存時間・転送時間を分けて集計する"""
    ms = result["metrics"]
    spec = result["spec"]
    out = dict(n_steps=len(ms))
    if not ms:
        return out
    saves = set(spec["expected_save_steps"])
    intervals = [(ms[i]["step"], ms[i]["t"] - ms[i - 1]["t"]) for i in range(1, len(ms))]
    plain = [d for s, d in intervals if (s - 1) not in saves]      # 直前 step で保存が走った区間を除く
    out["step_sec_median"] = float(np.median(plain)) if plain else None
    out["step_sec_all"] = [round(d, 2) for _, d in intervals]
    first_step_t = ms[0]["t"]
    out["init_sec_incl_first_step"] = first_step_t - result["launched_at"]
    out["init_sec_est"] = (first_step_t - result["launched_at"] - out["step_sec_median"]) if out["step_sec_median"] else None
    ev = result["events"]
    save_starts = [e["t"] for e in ev if "Saving checkpoint to" in e["line"]]
    save_ends = [e["t"] for e in ev if "Updated checkpoint tracker" in e["line"]]
    out["save_sec"] = [round(b - a, 1) for a, b in zip(save_starts, save_ends)]
    out["upload_sec"] = {s: round(v.get("duration_sec", 0), 1) for s, v in result["monitor"]["uploaded"].items()}
    out["gpu_mem_max_mb"] = result["monitor"]["gpu_mem_max_mb"]
    return out


# ---- HF からの再開 ------------------------------------------------------------------------------
def list_hf_checkpoints(run_id):
    """HF 上の run の checkpoint 一覧。完了マーカーのあるものだけを再開候補にする"""
    paths = hf_run_paths(HF_CKPT_REPO_ID, run_id)
    rows = []
    for step in sorted(paths):
        row = dict(step=step, complete=paths[step]["complete"], n_files=len(paths[step]["files"]))
        if row["complete"]:
            try:
                mp = hf_hub_download(HF_CKPT_REPO_ID, f"runs/{run_id}/global_step_{step}/ao_upload_verified.json", repo_type="model", local_dir=f"{WORK_DIR}/hf_markers")
                row.update(json.load(open(mp)))
            except Exception as e:  # noqa
                row["complete"] = False
                row["marker_error"] = str(e)[:200]
        rows.append(row)
    return rows


def download_hf_checkpoint(run_id, step, revision=None):
    """HF から checkpoint を取得し manifest で検証する。戻り値: (local_path, manifest, revision)"""
    pir = f"runs/{run_id}/global_step_{step}"
    marker_path = hf_hub_download(HF_CKPT_REPO_ID, f"{pir}/ao_upload_verified.json", repo_type="model", local_dir=f"{WORK_DIR}/hf_markers")
    marker = json.load(open(marker_path))
    revision = revision or marker["folder_commit"]
    local = f"{CKPT_DIR}/{run_id}/global_step_{step}"
    tmp = f"{WORK_DIR}/hf_download/{run_id}_{step}"
    shutil.rmtree(tmp, ignore_errors=True)
    snapshot_download(HF_CKPT_REPO_ID, repo_type="model", revision=revision, allow_patterns=[f"{pir}/*", f"{pir}/**"], local_dir=tmp)
    src = os.path.join(tmp, pir)
    if os.path.isdir(local):
        shutil.rmtree(local)
    os.makedirs(os.path.dirname(local), exist_ok=True)
    shutil.move(src, local)
    shutil.copy(marker_path, os.path.join(local, "ao_upload_verified.json"))   # マーカーは folder_commit より後の commit なので別途置く（監視が再転送しないための目印）
    manifest = json.load(open(os.path.join(local, "ao_ckpt_manifest.json")))
    bad = []
    for e in manifest["files"]:
        p = os.path.join(local, e["path"])
        if not os.path.isfile(p) or os.path.getsize(p) != e["size"]:
            bad.append((e["path"], "missing/size")); continue
        if e.get("sha256") and sha256_of(p) != e["sha256"]:
            bad.append((e["path"], "sha256"))
    if bad:
        raise RuntimeError(f"取得した checkpoint が manifest と一致しない: {bad[:5]}")
    # 公式の latest tracker も揃えておく（resume_path モードでは使われないが、ディレクトリ構成を学習時と同じにする）
    with open(os.path.join(os.path.dirname(local), "latest_checkpointed_iteration.txt"), "w") as f:
        f.write(str(step))
    ok, _ = checkpoint_is_complete(local, manifest["world_size"], require_stable=False)
    assert ok, "必要ファイル（model/optim/extra の各 rank shard, data.pt, fsdp_config.json, huggingface/）が揃っていない"
    manifest["folder_commit"] = marker.get("folder_commit")
    print(f"downloaded {pir} @ {revision[:10]} -> {local} ({manifest['total_bytes'] / 1e9:.2f} GB, verified against manifest)")
    return local, manifest, revision


def current_data_sha256_for(saved_data_path):
    """保存時の data_path に対応する現在のファイルの sha256（本学習は AO parquet、試走は subset parquet）"""
    if os.path.basename(saved_data_path) == os.path.basename(AO_PARQUET):
        return AO_SHA256, AO_PARQUET
    if os.path.isfile(saved_data_path):
        return sha256_of(saved_data_path), saved_data_path
    return None, None


def check_resume_compatibility(manifest, n_gpus=None):
    """コード・データ・tokenizer・学習設定・GPU 台数/FSDP 構成の互換性を確認し、問題を列挙する"""
    n_gpus = n_gpus or N_GPUS
    saved = manifest["run_config"]
    problems, warnings_ = [], []
    if manifest["world_size"] != n_gpus:
        problems.append(f"FSDP world_size {manifest['world_size']} != 現在の GPU 台数 {n_gpus}（sharded checkpoint は台数依存。条件を黙って変えない）")
    if saved["base_revision"] != MODEL_INFO["base_revision"] or saved["model_key"] != MODEL_KEY:
        problems.append(f"Base モデル/revision が異なる: saved={saved['model_key']}@{saved['base_revision'][:8]} now={MODEL_KEY}@{MODEL_INFO['base_revision'][:8]}")
    cur_sha, cur_path = current_data_sha256_for(saved["data_path"])
    if cur_sha is None:
        problems.append(f"学習データ {saved['data_path']} が現在の環境に無い（セクション 3 / 試走セルで再作成する）")
    elif saved["data_sha256"] != cur_sha:
        problems.append(f"学習データの sha256 が異なる: saved={saved['data_sha256'][:12]} now={cur_sha[:12]} ({cur_path})")
    if saved["target_style"] != AO_TARGET_STYLE:
        problems.append("AO_TARGET_STYLE が異なる")
    if saved.get("fork_commit") != FORK_COMMIT:
        if OFFICIAL_CODE_UNCHANGED is True:
            warnings_.append(f"Fork commit が異なる: saved={str(saved.get('fork_commit'))[:10]} now={FORK_COMMIT[:10]}（verl/・training_scripts/ は参照 commit と同一なので続行可）")
        else:
            problems.append(f"Fork commit が異なり、公式コードの無変更を確認できない (OFFICIAL_CODE_UNCHANGED={OFFICIAL_CODE_UNCHANGED})")
    if saved["kind"] != "trial" and "trainer.total_training_steps" in saved["overrides"]:
        problems.append("保存時の設定に step 上限 (trainer.total_training_steps) がある。本学習の checkpoint ではない")
    if saved.get("max_length") != int(MAX_LENGTH):
        warnings_.append(f"現在の MAX_LENGTH {MAX_LENGTH} と保存時 {saved.get('max_length')} が異なる。再開は保存時の設定を使う")
    if not FLASH_ATTN_OK and n_gpus > 0:
        problems.append("flash-attn が使えない")
    return problems, warnings_


def build_resume_spec(manifest, local_ckpt_path):
    """保存時の run 設定をそのまま使い、resume 引数だけを付けた spec を作る（総 epoch 数・総 step 数を維持）"""
    saved = manifest["run_config"]
    spec = json.loads(json.dumps(saved))
    ov = spec["overrides"]
    ov["trainer.resume_mode"] = "resume_path"
    ov["trainer.resume_from_path"] = local_ckpt_path
    ov["model.partial_pretrain"] = get_base_model_local()          # 同じ revision を再取得（パスは環境で変わり得る）
    cur_sha, cur_path = current_data_sha256_for(saved["data_path"])
    assert cur_sha == saved["data_sha256"], "学習データが保存時と一致しない"
    ov["data.train_files"] = cur_path
    ov["data.val_files"] = ov["data.train_files"]
    ov["trainer.default_local_dir"] = f"{CKPT_DIR}/{spec['run_id']}"
    spec["base_local"] = ov["model.partial_pretrain"]
    spec["data_path"] = ov["data.train_files"]
    spec["resume_from_path"] = local_ckpt_path
    spec["resumed_from"] = dict(step=manifest["step"], hf_repo=HF_CKPT_REPO_ID, folder_commit=manifest.get("folder_commit"))
    spec["fork_commit_at_resume"] = FORK_COMMIT
    spec["env"]["VERL_SFT_LOGGING_LEVEL"] = "INFO"
    assert ov["trainer.total_epochs"] == str(saved["epochs"]) and spec["total_steps"] == saved["total_steps"], "総 epoch 数・総 step 数は保存時の値を維持する"
    return spec


def resume_run_from_hf(run_id, step="latest", kill_after_step=None, upload=None):
    """HF の完了済み checkpoint から新プロセスで再開する。未保存 step は再実行になる"""
    rows = list_hf_checkpoints(run_id)
    complete = [r for r in rows if r["complete"]]
    print(f"HF checkpoints for {run_id}:")
    for r in rows:
        print(f"  step {r['step']:6d} complete={r['complete']} files={r['n_files']} commit={str(r.get('folder_commit', ''))[:10]} verified_at={r.get('verified_at', '')}")
    if not complete:
        raise SystemExit("完了済み checkpoint が無い。再開できない（不完全な転送は候補にしない）")
    if step == "latest":
        chosen = max(r["step"] for r in complete)
    else:
        chosen = int(step)
        assert chosen in [r["step"] for r in complete], f"step {chosen} は完了済みではない"
        newer = [r["step"] for r in complete if r["step"] > chosen]
        if newer:
            raise SystemExit(f"step {chosen} より新しい完了済み step {newer} がある。古い step から再開すると新しい step を上書きする恐れがあるので拒否する。"
                             "別 run として続ける場合は run_id を変えて手動で設定する")
    local, manifest, rev = download_hf_checkpoint(run_id, chosen)
    problems, warns = check_resume_compatibility(manifest)
    for w in warns:
        print("WARN:", w)
    if problems:
        for p in problems:
            print("PROBLEM:", p)
        raise SystemExit("再開条件を満たさない。条件を黙って変更しない")
    spec = build_resume_spec(manifest, local)
    total = spec["total_steps"]
    print(f"resume {run_id} from global_step_{chosen} (HF revision {rev[:10]}). 総 step {total} は維持。step {chosen + 1}..{total} を再実行（最後の保存以降の未保存 step は再実行になる）")
    print_run_spec(spec)
    return launch_training(spec, kill_after_step=kill_after_step, upload=upload)


# ---- FSDP checkpoint → HF 形式（公式 verl.model_merger をそのまま使う） ---------------------------
def merge_checkpoint(run_id, step, local_ckpt=None):
    """python -m verl.model_merger merge --backend fsdp（README の手順）で HF 形式へ変換する。戻り値: 変換先ディレクトリ"""
    ckpt = local_ckpt or f"{CKPT_DIR}/{run_id}/global_step_{step}"
    ok = False
    if os.path.isfile(os.path.join(ckpt, "fsdp_config.json")):
        world = json.load(open(os.path.join(ckpt, "fsdp_config.json")))["world_size"]
        ok, _ = checkpoint_is_complete(ckpt, world, require_stable=False)
    if not ok:   # ローカルに無い、または書きかけ → HF の完了済み checkpoint を取得する
        ckpt, _, _ = download_hf_checkpoint(run_id, step)
        world = json.load(open(os.path.join(ckpt, "fsdp_config.json")))["world_size"]
        ok, _ = checkpoint_is_complete(ckpt, world, require_stable=False)
    assert ok, f"checkpoint が不完全: {ckpt}"
    target = f"{WORK_DIR}/merged/{run_id}/merged_step{step}"
    if os.path.isfile(os.path.join(target, "config.json")) and glob.glob(os.path.join(target, "*.safetensors")):
        print("merged model exists:", target)
        return target
    os.makedirs(target, exist_ok=True)
    cmd = [sys.executable, "-m", "verl.model_merger", "merge", "--backend", "fsdp", "--local_dir", ckpt, "--target_dir", target, "--trust_remote_code"]
    env = dict(os.environ, PYTHONPATH=REPO_DIR + os.pathsep + os.environ.get("PYTHONPATH", ""))
    t0 = time.time()
    r = subprocess.run(cmd, cwd=REPO_DIR, env=env, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-4000:])
        raise RuntimeError("verl.model_merger failed")
    assert os.path.isfile(os.path.join(target, "config.json")) and glob.glob(os.path.join(target, "*.safetensors")), "変換結果に config/safetensors が無い"
    assert any(os.path.isfile(os.path.join(target, f)) for f in ("tokenizer.json", "tokenizer_config.json")), "変換結果に tokenizer が無い"
    print(f"merged {ckpt} -> {target} ({time.time() - t0:.0f}s)")
    return target


def get_manifest(run_id, step):
    """checkpoint の manifest（保存時の run 設定を含む）。ローカルに無ければ HF から manifest だけ取得する"""
    local = f"{CKPT_DIR}/{run_id}/global_step_{step}/ao_ckpt_manifest.json"
    if os.path.isfile(local):
        return json.load(open(local))
    p = hf_hub_download(HF_CKPT_REPO_ID, f"runs/{run_id}/global_step_{step}/ao_ckpt_manifest.json", repo_type="model", local_dir=f"{WORK_DIR}/hf_markers")
    return json.load(open(p))


# ---- 実験記録 ----------------------------------------------------------------------------------
def write_experiment_record(spec, results, evaluations=None, final_model=None, unverified=None, upload=True):
    rec = dict(
        run_id=spec["run_id"], kind=spec["kind"], written_at=now_iso(), session_started_at=SESSION_STARTED_AT,
        models=dict(teacher=TEACHER, base=dict(repo=spec["base_repo"], revision=spec["base_revision"]),
                    cot_student=dict(repo=spec["cot_repo"], revision=spec["cot_revision"], subfolder=spec["cot_subfolder"])),
        data=dict(repo=spec["dataset_repo"], revision=spec["dataset_revision"], raw_sha256=DATASET_EXPECTED_SHA256, ao_parquet_sha256=spec["data_sha256"],
                  rows=spec["n_rows"], target_style=spec["target_style"], ao_audit=AO_AUDIT_SUMMARY, prompt_note="prompt は公式と同一（step by step の指示を含む）。target は最後の \\boxed{} のみ"),
        length_and_mask=LENGTH_RECORD, training=dict(overrides=spec["overrides"], changes_vs_official=spec["changes_vs_official"], env=spec["env"],
                                                     total_steps=spec["total_steps"], steps_per_epoch=spec["steps_per_epoch"], warmup_steps=spec["warmup_steps"],
                                                     save_freq=spec["save_freq"], official_script=spec["official_script"], n_gpus=spec["n_gpus"]),
        code=dict(fork_repo=FORK_REPO_URL, fork_commit=FORK_COMMIT, upstream_reference_commit=UPSTREAM_REFERENCE_COMMIT, official_code_unchanged=OFFICIAL_CODE_UNCHANGED),
        environment=ENV_RECORD,
        runs=[dict(exit_code=r["exit_code"], killed_by_monitor=r["killed_by_monitor"], log_path=r["log_path"], wall_sec=r["wall_sec"], n_steps_logged=len(r["metrics"]),
                   first_step=(r["metrics"][0]["step"] if r["metrics"] else None), last_step=(r["metrics"][-1]["step"] if r["metrics"] else None),
                   last_loss=(r["metrics"][-1].get("train/loss") if r["metrics"] else None), timing=summarize_timing(r),
                   checkpoints_uploaded={s: dict(commit=v.get("commit"), marker_commit=v.get("marker_commit"), url=v.get("url")) for s, v in r["monitor"]["uploaded"].items()},
                   upload_failures=r["monitor"]["failed"], resumed_from=r["spec"].get("resumed_from")) for r in results],
        evaluations=evaluations or [], final_model=final_model,
        unverified=unverified or [],
        caveat="loss 低下だけで能力向上は主張しない。評価欄が空なら性能は未測定。",
    )
    path = f"{RECORD_DIR}/experiment_record_{spec['run_id']}.json"
    with open(path, "w") as f:
        json.dump(rec, f, indent=2, ensure_ascii=False, default=str)
    print("experiment record:", path)
    if upload and HF_API is not None and HF_CKPT_REPO_ID:
        ensure_ckpt_repo()
        ts = time.strftime("%Y%m%d-%H%M%S")
        for name in (f"experiment_record_{ts}.json", "experiment_record_latest.json"):
            HF_API.upload_file(path_or_fileobj=path, path_in_repo=f"runs/{spec['run_id']}/{name}", repo_id=HF_CKPT_REPO_ID, repo_type="model",
                               commit_message=f"{spec['run_id']} experiment record {ts}")
        print(f"uploaded to https://huggingface.co/{HF_CKPT_REPO_ID}/tree/main/runs/{spec['run_id']}")
    return rec


print("helpers ready: launch_training / TrainingMonitor / upload_checkpoint / list_hf_checkpoints / download_hf_checkpoint / resume_run_from_hf / write_experiment_record")

In [ ]:
# @title 6-b. 短い試走: 学習 → HF 保存 → プロセス終了 → HF から取得 → 新プロセスで再開（RUN_MODE="trial" のときだけ実行）
import pyarrow.parquet as pq, json, os, math, numpy as np

TRIAL_RESULTS = {}
if RUN_MODE != "trial":
    print("RUN_MODE != 'trial' なのでスキップ")
else:
    assert N_GPUS >= 1 and FLASH_ATTN_OK, "GPU と flash-attn が必要"
    assert AO_DATA_READY, "AO データが未確定（セクション 3 のゲート）。試走でも同じデータ条件を使う"
    # 試走データ: AO parquet の先頭 TRIAL_NUM_ROWS 行（行順維持）。本学習と混ざらないよう別ファイル・別 run_id
    _trial_rows = (TRIAL_NUM_ROWS // OFFICIAL_TBS) * OFFICIAL_TBS
    assert _trial_rows >= OFFICIAL_TBS, "TRIAL_NUM_ROWS は global batch 256 以上"
    TRIAL_PARQUET = f"{DATA_DIR}/trial/Math-AO-trial{_trial_rows}.parquet"
    os.makedirs(os.path.dirname(TRIAL_PARQUET), exist_ok=True)
    pq.write_table(AO_TABLE.slice(0, _trial_rows), TRIAL_PARQUET)
    TRIAL_SHA256 = sha256_of(TRIAL_PARQUET)
    _ts = time.strftime("%Y%m%d-%H%M%S")
    _trial_id = f"trial-{MODEL_KEY}_Math-AO-{_trial_rows}_lr{fmt_lr(OFFICIAL_BASELINE['lr'])}_ep{TRIAL_EPOCHS}_bs{OFFICIAL_TBS}_{_ts}"
    _spe = steps_per_epoch(_trial_rows, OFFICIAL_TBS, N_GPUS)
    _total = _spe * TRIAL_EPOCHS
    assert TRIAL_KILL_AFTER_STEP in range(TRIAL_SAVE_FREQ, _total, TRIAL_SAVE_FREQ), f"TRIAL_KILL_AFTER_STEP は保存 step（{TRIAL_SAVE_FREQ} の倍数）かつ最終 step {_total} 未満"
    TRIAL_SPEC = build_run_spec("trial", OFFICIAL_BASELINE["lr"], TRIAL_EPOCHS, data_path=TRIAL_PARQUET, data_sha256=TRIAL_SHA256, n_rows=_trial_rows,
                                max_length=MAX_LENGTH, save_freq=TRIAL_SAVE_FREQ, run_id=_trial_id,
                                extra_note=f"試走。データ {_trial_rows} 行 / {TRIAL_EPOCHS} epoch / step 上限なし（scheduler 期間 = {_total} step）。本学習の設定ではない")
    print_run_spec(TRIAL_SPEC)
    print(f"\n[試走 A] step {TRIAL_KILL_AFTER_STEP} の checkpoint 保存完了を検知 → 学習プロセスを一時停止 → HF へ転送・検証 → プロセス終了（切断の模擬）")
    TRIAL_RESULTS["A_interrupted"] = launch_training(TRIAL_SPEC, kill_after_step=TRIAL_KILL_AFTER_STEP, upload=True, poll_sec=5, stable_scans=False)
    _a = TRIAL_RESULTS["A_interrupted"]
    assert _a["killed_by_monitor"], "監視スレッドによる終了が起きていない（プロセスが先に完走した、または HF 転送が完了していない）"
    assert TRIAL_KILL_AFTER_STEP in _a["monitor"]["uploaded"], "kill step の checkpoint が HF に転送・検証されていない"
    _a_last = max(m["step"] for m in _a["metrics"])
    assert _a_last < _total, f"run A が最終 step {_total} まで進んでしまい、中断の模擬になっていない（last step {_a_last}）"
    print(f"run A: step {_a_last} で終了（総 step {_total}）。step {TRIAL_KILL_AFTER_STEP + 1}〜{_a_last} は再実行される")
    _hf_before = {r["step"]: r.get("folder_commit") for r in list_hf_checkpoints(_trial_id) if r["complete"]}

    # ローカル checkpoint を消して、HF からの取得だけで再開できることを確かめる
    shutil.rmtree(f"{CKPT_DIR}/{_trial_id}", ignore_errors=True)
    print(f"\n[試走 B] ローカル checkpoint を削除し、HF から step {TRIAL_KILL_AFTER_STEP} を取得して新プロセスで再開（総 epoch {TRIAL_EPOCHS} / 総 step {_total} を維持）")
    TRIAL_RESULTS["B_resumed"] = resume_run_from_hf(_trial_id, step=TRIAL_KILL_AFTER_STEP, upload=True)
    _b = TRIAL_RESULTS["B_resumed"]
    assert _b["exit_code"] == 0, "再開 run が異常終了"
    _b_steps = [m["step"] for m in _b["metrics"]]
    assert _b_steps and _b_steps[0] == TRIAL_KILL_AFTER_STEP + 1 and _b_steps[-1] == _total, f"再開後の step 列が想定外: {_b_steps}"
    _b_log = open(_b["log_path"]).read()
    _hf_after = {r["step"]: r.get("folder_commit") for r in list_hf_checkpoints(_trial_id) if r["complete"]}
    RESUME_CHECKS = dict(
        global_step_continued=(_b_steps[0] == TRIAL_KILL_AFTER_STEP + 1),
        model_loaded=("Loaded model from" in _b_log), optimizer_loaded=("Loaded optimizer from" in _b_log),
        lr_scheduler_loaded=("Loaded lr_scheduler from" in _b_log), rng_loaded=("Loaded rng from" in _b_log),
        dataloader_state_loaded=("Successfully loaded dataloader state" in _b_log),
        final_ckpt_uploaded=(_total in _b["monitor"]["uploaded"] and not _b["monitor"]["uploaded"][_total].get("previously_verified")),
        kill_step_ckpt_not_overwritten=(_hf_before.get(TRIAL_KILL_AFTER_STEP) == _hf_after.get(TRIAL_KILL_AFTER_STEP) is not None),
        no_upload_failures=(not _b["monitor"]["failed"]),
    )
    print("再開チェック:", RESUME_CHECKS)

    # 解析的な cosine schedule と比較（warmup = int(total*0.1)、verl get_cosine_schedule_with_warmup と同式）
    def expected_lr(step, total, base_lr, warmup_ratio=float(YAML_DEFAULTS.optim.warmup_steps_ratio)):
        warm = int(total * warmup_ratio)
        if step < warm:
            return base_lr * step / max(1, warm)
        prog = (step - warm) / max(1, total - warm)
        return base_lr * max(0.0, 0.5 * (1 + math.cos(math.pi * prog)))

    LR_CHECK = []
    for m in _b["metrics"]:
        exp = expected_lr(m["step"], _total, OFFICIAL_BASELINE["lr"])   # train/lr は step 後の get_last_lr
        LR_CHECK.append(dict(step=m["step"], logged=m.get("train/lr"), expected_after_step=exp, ok=abs(m.get("train/lr", -1) - exp) <= 1e-9 + 1e-3 * abs(exp)))
    print("lr の継続（再開後）:", LR_CHECK)
    RESUME_CHECKS["lr_schedule_continued"] = all(x["ok"] for x in LR_CHECK)

    if TRIAL_RUN_REFERENCE:
        print(f"\n[試走 C] 中断なしの参照 run（同じ設定）。再開後の loss / データ位置を比較する")
        _ref_spec = build_run_spec("trial", OFFICIAL_BASELINE["lr"], TRIAL_EPOCHS, data_path=TRIAL_PARQUET, data_sha256=TRIAL_SHA256, n_rows=_trial_rows,
                                   max_length=MAX_LENGTH, save_freq=TRIAL_SAVE_FREQ, run_id=_trial_id + "-ref", extra_note="試走の参照 run（中断なし）")
        TRIAL_RESULTS["C_reference"] = launch_training(_ref_spec, upload=False, keep_local=99)
        _c = {m["step"]: m for m in TRIAL_RESULTS["C_reference"]["metrics"]}
        _cmp = []
        for m in _b["metrics"]:
            r = _c.get(m["step"])
            if r:
                _cmp.append(dict(step=m["step"], loss_resumed=m.get("train/loss"), loss_reference=r.get("train/loss"),
                                 rel_diff=abs(m.get("train/loss", 0) - r.get("train/loss", 0)) / max(1e-8, abs(r.get("train/loss", 0))),
                                 lr_equal=abs(m.get("train/lr", 0) - r.get("train/lr", 0)) < 1e-12))
        print("再開 run と参照 run の比較（同じ step で同じバッチなら loss は近い。大きくずれればデータ位置が復元されていない）:")
        for x in _cmp:
            print("  ", x)
        RESUME_CHECKS["loss_matches_reference_within_5pct"] = all(x["rel_diff"] < 0.05 for x in _cmp) if _cmp else None
        RESUME_CHECKS["lr_equals_reference"] = all(x["lr_equal"] for x in _cmp) if _cmp else None

    # ---- 時間計測と本学習の概算 ----
    _tA, _tB = summarize_timing(_a), summarize_timing(_b)
    _tC = summarize_timing(TRIAL_RESULTS["C_reference"]) if TRIAL_RUN_REFERENCE else None
    _step_sec = np.median([x for t in (_tA, _tB, _tC) if t and t.get("step_sec_median") for x in [t["step_sec_median"]]])
    _init_sec = _tA.get("init_sec_est") or _tA.get("init_sec_incl_first_step")
    _save_sec = float(np.median([s for t in (_tA, _tB) for s in t.get("save_sec", [])])) if any(t.get("save_sec") for t in (_tA, _tB)) else None
    _upload_sec = float(np.median([s for t in (_tA, _tB) for s in t.get("upload_sec", {}).values()])) if any(t.get("upload_sec") for t in (_tA, _tB)) else None
    print("\n=== 試走の計測 ===")
    print(f"  初期化（起動〜最初の step 前）≈ {_init_sec:.0f}s | optimizer step（勾配蓄積 {TRIAL_SPEC['grad_accum_micro_batches']} micro batch 込み）中央値 {_step_sec:.1f}s")
    print(f"  checkpoint 保存 {_save_sec}s | HF 転送+検証 {_upload_sec}s（バックグラウンド） | GPU メモリ最大 {_a['monitor']['gpu_mem_max_mb']} MB")
    _base_steps = steps_per_epoch(DATASET_EXPECTED_ROWS, OFFICIAL_TBS, N_GPUS) * OFFICIAL_BASELINE["epochs"]
    _base_saves = len(set(list(range(SAVE_FREQ, _base_steps + 1, SAVE_FREQ)) + [_base_steps]))
    _base_est = _init_sec + _base_steps * _step_sec + _base_saves * (_save_sec or 0)
    print(f"  baseline 概算: init {_init_sec:.0f}s + {_base_steps} steps x {_step_sec:.1f}s + {_base_saves} saves x {_save_sec or 0}s ≈ {_base_est / 3600:.1f} h（HF 転送は並行。評価・変換・最終アップロードは未計測）")
    _grid = SEARCH_GRID_CONFIRMED or SEARCH_GRID_PROPOSED
    _grid_steps = sum(steps_per_epoch(DATASET_EXPECTED_ROWS, OFFICIAL_TBS, N_GPUS) * e for e in _grid["epochs"]) * len(_grid["lr"])
    print(f"  探索全体（{len(_grid['lr'])} lr x {len(_grid['epochs'])} epoch 設定 = {_grid_steps} steps）概算 ≈ {(_grid_steps * _step_sec + len(_grid['lr']) * len(_grid['epochs']) * _init_sec) / 3600:.1f} h（保存・評価は含まない）")
    print("  注意: 試走は step 上限を使わず epoch 数で短くしたので scheduler 期間は試走内で完結している。試走の checkpoint・重み・データ削減は本学習に使わない")
    TRIAL_TIMING = dict(init_sec=_init_sec, step_sec_median=float(_step_sec), save_sec=_save_sec, upload_sec=_upload_sec, gpu_mem_max_mb=_a["monitor"]["gpu_mem_max_mb"],
                        baseline_est_hours=_base_est / 3600, timing_A=_tA, timing_B=_tB, timing_C=_tC, resume_checks=RESUME_CHECKS, lr_check=LR_CHECK)
    with open(f"{RECORD_DIR}/trial_{_trial_id}.json", "w") as f:
        json.dump(dict(spec=TRIAL_SPEC, timing=TRIAL_TIMING, results={k: dict(exit_code=v["exit_code"], log=v["log_path"], uploaded=list(v["monitor"]["uploaded"])) for k, v in TRIAL_RESULTS.items()}),
                  f, indent=2, ensure_ascii=False, default=str)
    if not all(v for v in RESUME_CHECKS.values() if v is not None):
        print("*** 再開チェックに失敗項目がある。原因を解決するまで本学習へ進まない ***")
    else:
        print("試走の再開チェックはすべて合格")

## 7. baseline の確認または実行

In [ ]:
# @title 7. baseline（公式 CoT 設定 lr 5e-5 / 8 epoch の AO 学習）の確認または実行
TRAIN_RESULTS = globals().get("TRAIN_RESULTS", {})


def preflight_train(spec):
    """本学習の直前確認: データ条件・ゲート・試走設定の混入・衝突"""
    assert RUN_MODE == "train", "RUN_MODE='train' でのみ本学習を開始する"
    assert N_GPUS >= 1 and FLASH_ATTN_OK, "GPU と flash-attn が必要"
    assert AO_DATA_READY, "AO データに未解決の抽出/照合問題がある（セクション 3）"
    assert MASK_CHECK_OK, "loss mask の確認に失敗している（セクション 4）"
    assert spec["data_path"] == AO_PARQUET and spec["n_rows"] == DATASET_EXPECTED_ROWS and spec["data_sha256"] == AO_SHA256, "本学習は全 20,480 行の AO データで行う"
    assert not spec["run_id"].startswith("trial-") and spec["total_training_steps_override"] is None, "試走設定が本学習に混入している"
    assert "trainer.total_training_steps" not in spec["overrides"], "step 上限は本学習で使わない"
    assert spec["overrides"]["trainer.resume_mode"] == "disable", "新規 run は同じ Base から独立に開始する（再開は『再開』セル）"
    if HF_UPLOAD_CHECKPOINTS:
        assert HF_API is not None, "HF 未ログイン。学習中の HF 保存が要件なので開始しない"
        remote = hf_run_paths(HF_CKPT_REPO_ID, spec["run_id"])
        assert not remote, f"HF に同名 run {spec['run_id']} が存在する（steps {sorted(remote)}）。上書きしない。再開は『再開』セル、別 run は RUN_ID_SUFFIX"
    assert not os.path.exists(spec["overrides"]["trainer.default_local_dir"]), f"ローカルに同名 run のディレクトリがある: {spec['overrides']['trainer.default_local_dir']}"
    _disk = shutil.disk_usage(WORK_DIR)
    print(f"preflight OK: disk free {_disk.free / 1e9:.0f} GB (ckpt ≈ {CKPT_SIZE_EST_GB:.1f} GB x local keep {LOCAL_KEEP_LAST_N_VERIFIED_CKPTS})")


if RUN_MODE != "train" or RUN_KIND != "baseline":
    print("RUN_MODE='train' かつ RUN_KIND='baseline' のときだけ実行する（現在: %s / %s）" % (RUN_MODE, RUN_KIND))
    if HF_API is not None and HF_CKPT_REPO_ID:
        _bid = make_run_id("baseline", OFFICIAL_BASELINE["lr"], OFFICIAL_BASELINE["epochs"], suffix=RUN_ID_SUFFIX)
        _remote = hf_run_paths(HF_CKPT_REPO_ID, _bid)
        print(f"HF 上の baseline run {_bid}: {'steps ' + str(sorted(_remote)) if _remote else 'なし'}")
        print("  注意: 既存成果物があっても、その run_config（manifest）で AO target と設定を確認するまで再利用可能とは判断しない")
else:
    BASELINE_SPEC = build_run_spec("baseline", OFFICIAL_BASELINE["lr"], OFFICIAL_BASELINE["epochs"], data_path=AO_PARQUET, data_sha256=AO_SHA256,
                                   n_rows=DATASET_EXPECTED_ROWS, max_length=MAX_LENGTH, save_freq=SAVE_FREQ,
                                   extra_note="baseline: 公開 CoT 設定 (lr 5e-5, 8 epoch) を AO データで実行")
    print_run_spec(BASELINE_SPEC)
    preflight_train(BASELINE_SPEC)
    TRAIN_RESULTS[BASELINE_SPEC["run_id"]] = launch_training(BASELINE_SPEC)
    _r = TRAIN_RESULTS[BASELINE_SPEC["run_id"]]
    print("timing:", json.dumps({k: v for k, v in summarize_timing(_r).items() if k != "step_sec_all"}, default=str))
    write_experiment_record(BASELINE_SPEC, [_r], unverified=["dev 評価未実施", "最終モデル未変換"])

## 8. LR / epoch 探索

In [ ]:
# @title 8. LR / epoch 探索（各候補を同じ Base から独立に学習。8 epoch run の途中 checkpoint を短い epoch 設定の代用にしない）
TRAIN_RESULTS = globals().get("TRAIN_RESULTS", {})
SEARCH_SPECS = {}
if RUN_MODE != "train" or RUN_KIND != "search":
    print("RUN_MODE='train' かつ RUN_KIND='search' のときだけ実行する（現在: %s / %s）" % (RUN_MODE, RUN_KIND))
    print("提案グリッド:", SEARCH_GRID_PROPOSED, "| 確定グリッド:", SEARCH_GRID_CONFIRMED, "| 実行リスト:", SEARCH_RUN_LIST)
elif not SEARCH_RUN_LIST:
    print("SEARCH_RUN_LIST が空。実行する (lr, epochs) を設定セルで指定する")
else:
    _grid = SEARCH_GRID_CONFIRMED or SEARCH_GRID_PROPOSED
    for _lr, _ep in SEARCH_RUN_LIST:
        _tag = "confirmed" if (SEARCH_GRID_CONFIRMED and _lr in SEARCH_GRID_CONFIRMED["lr"] and _ep in SEARCH_GRID_CONFIRMED["epochs"]) else "proposed"
        _spec = build_run_spec("search", _lr, _ep, data_path=AO_PARQUET, data_sha256=AO_SHA256, n_rows=DATASET_EXPECTED_ROWS, max_length=MAX_LENGTH,
                               save_freq=SAVE_FREQ, extra_note=f"探索候補 ({_tag} grid)。総 epoch {_ep} に応じて scheduler 期間 = {steps_per_epoch(DATASET_EXPECTED_ROWS, OFFICIAL_TBS, N_GPUS) * _ep} step")
        SEARCH_SPECS[_spec["run_id"]] = _spec
    for _rid, _spec in SEARCH_SPECS.items():
        print("=" * 100)
        print_run_spec(_spec)
        _remote = hf_run_paths(HF_CKPT_REPO_ID, _rid) if HF_API else {}
        if _remote:
            print(f"HF に {_rid} が既にある (steps {sorted(_remote)})。完了済みなら再実行しない。途中なら『再開』セルで RESUME_RUN_ID を指定する")
            continue
        preflight_train(_spec)
        TRAIN_RESULTS[_rid] = launch_training(_spec)
        write_experiment_record(_spec, [TRAIN_RESULTS[_rid]], unverified=["dev 評価未実施", "最終モデル未変換"])
    print("探索 run の完了状況:", {k: ("OK" if v["exit_code"] == 0 else f"rc={v['exit_code']}") for k, v in TRAIN_RESULTS.items()})

## 再開: Colab 切断後に HF の checkpoint から続ける

切断後は **セル 0（認証）→ 設定セル → 1 → 2 → 3 → 4-a → 4-b → 6-a** を実行してから、`RESUME_RUN_ID` を設定してこのセルを実行する。

In [ ]:
# @title 再開: Colab 切断後に HF の完了済み checkpoint から再開する（RESUME_RUN_ID を設定。セル 0〜4 と 6-a を先に実行）
# 手順: 1) 環境と Fork の commit を復元（セル 2） 2) HF の checkpoint 一覧 3) 最新の完了済み or 指定 step を選択
#       4) HF revision を記録して取得・manifest で検証 5) trainer.resume_mode=resume_path / resume_from_path で再開
TRAIN_RESULTS = globals().get("TRAIN_RESULTS", {})
if not RESUME_RUN_ID:
    print("RESUME_RUN_ID が未設定。再開しない。HF 上の run 一覧:")
    if HF_API is not None and HF_CKPT_REPO_ID and hf_repo_exists(HF_CKPT_REPO_ID):
        _runs = sorted({f.split("/")[1] for f in HF_API.list_repo_files(HF_CKPT_REPO_ID, repo_type="model") if f.startswith("runs/")})
        for _r in _runs:
            print("  ", _r)
else:
    assert RUN_MODE in ("train", "trial"), "再開は RUN_MODE='train'（本学習）または 'trial'（試走 run）で行う"
    assert AO_DATA_READY and MASK_CHECK_OK
    TRAIN_RESULTS[RESUME_RUN_ID] = resume_run_from_hf(RESUME_RUN_ID, step=RESUME_STEP)
    _r = TRAIN_RESULTS[RESUME_RUN_ID]
    print("timing:", json.dumps({k: v for k, v in summarize_timing(_r).items() if k != "step_sec_all"}, default=str))
    write_experiment_record(_r["spec"], [_r], unverified=["再開 run。それ以前の run 結果は HF の experiment_record を参照"])

## 9. dev 評価と結果集計

In [ ]:
# @title 9. dev 評価と結果集計（既存の評価 parser + math-verify を流用。dev 未確定なら選択処理の準備に留める）
import importlib.util, json, os, sys, time, torch, gc
from math_verify import parse as mv_parse, verify as mv_verify
from transformers import AutoTokenizer, AutoModelForCausalLM

# 既存の評価コード evaluation/math_eval/utils/parser.py の extract_answer を読む（math_eval_budget.py と同じ抽出）
_ME_DIR = f"{REPO_DIR}/evaluation/math_eval"
if _ME_DIR not in sys.path:
    sys.path.insert(0, _ME_DIR)
_spec_p = importlib.util.spec_from_file_location("math_eval_parser", f"{_ME_DIR}/utils/parser.py")
_parser_mod = importlib.util.module_from_spec(_spec_p)
_spec_p.loader.exec_module(_parser_mod)
extract_answer = _parser_mod.extract_answer


def build_eval_prompt(question):
    """公式評価 (evaluation/math_eval/math_eval_budget.py build_prompt) と同じ"""
    return [{"role": "user", "content": "{}\n Please reason step by step, and put your final answer within \\boxed{{}}.".format(question)}]


def load_dev_rows():
    if not DEV_EVAL_SOURCE:
        return None
    rows = []
    if os.path.isfile(DEV_EVAL_SOURCE):
        with open(DEV_EVAL_SOURCE) as f:
            for line in f:
                if line.strip():
                    rows.append(json.loads(line))
    else:
        from datasets import load_dataset
        ds = load_dataset(DEV_EVAL_SOURCE, split=DEV_EVAL_SPLIT, revision=DEV_EVAL_REVISION)
        rows = [dict(r) for r in ds]
    out = [dict(question=r[DEV_EVAL_QUESTION_KEY], answer=(r[DEV_EVAL_ANSWER_KEY][0] if isinstance(r[DEV_EVAL_ANSWER_KEY], list) else r[DEV_EVAL_ANSWER_KEY])) for r in rows]
    return out[:DEV_EVAL_MAX_ROWS] if DEV_EVAL_MAX_ROWS else out


def score_responses(rows, responses):
    """math_eval_budget.py の process_single_result_no_truncation と同じ照合（gold を \\boxed{} で包み、応答全体を parse して verify）"""
    per = []
    for r, resp in zip(rows, responses):
        gold = str(r["answer"])
        gold_parsed = mv_parse(gold) if "boxed" in gold else mv_parse("\\boxed{" + gold + "}")
        try:
            pred = mv_parse(resp)
        except Exception:  # noqa
            pred = resp
        try:
            ok = bool(mv_verify(gold_parsed, pred))
        except Exception:  # noqa
            ok = False
        per.append(dict(gold=gold, extracted=extract_answer(resp, data_name="math", use_last_number=False), correct=ok, n_chars=len(resp)))
    acc = sum(p["correct"] for p in per) / max(1, len(per))
    return acc, per


def generate_hf(path_or_repo, prompts, subfolder=None, revision=None, max_new_tokens=512, batch_size=16, do_sample=False):
    kw = dict(trust_remote_code=True)
    if subfolder:      # ローカルディレクトリに subfolder=None を渡すと transformers 4.52 は失敗するので、指定時だけ渡す
        kw["subfolder"] = subfolder
    if revision:
        kw["revision"] = revision
    tok = AutoTokenizer.from_pretrained(path_or_repo, **kw)
    tok.padding_side = "left"
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(path_or_repo, torch_dtype=torch.bfloat16, **kw).cuda().eval()
    texts = [tok.apply_chat_template(p, add_generation_prompt=True, tokenize=False) for p in prompts]
    outs = []
    gen_kwargs = dict(max_new_tokens=max_new_tokens, do_sample=do_sample, eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id)
    if do_sample:
        gen_kwargs.update(temperature=0.6, top_p=0.95)   # 公式評価のサンプリング設定
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            enc = tok(texts[i:i + batch_size], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
            gen = model.generate(**enc, **gen_kwargs)
            outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    del model
    gc.collect(); torch.cuda.empty_cache()
    return outs


def evaluate_target(target, rows):
    t0 = time.time()
    prompts = [build_eval_prompt(r["question"]) for r in rows]
    resps = generate_hf(target["path_or_repo"], prompts, subfolder=target.get("subfolder"), revision=target.get("revision"),
                        max_new_tokens=target.get("max_new_tokens", DEV_EVAL_MAX_NEW_TOKENS), batch_size=DEV_EVAL_BATCH_SIZE, do_sample=DEV_EVAL_DO_SAMPLE)
    acc, per = score_responses(rows, resps)
    res = dict(name=target["name"], path_or_repo=target["path_or_repo"], subfolder=target.get("subfolder"), revision=target.get("revision"),
               dev_source=DEV_EVAL_SOURCE, dev_split=DEV_EVAL_SPLIT, dev_revision=DEV_EVAL_REVISION, n=len(rows), accuracy=acc, do_sample=DEV_EVAL_DO_SAMPLE,
               max_new_tokens=target.get("max_new_tokens", DEV_EVAL_MAX_NEW_TOKENS), elapsed_sec=time.time() - t0, scorer="math_eval_budget.py と同じ math-verify 照合 (greedy 1 サンプル)")
    os.makedirs(f"{RECORD_DIR}/dev_eval", exist_ok=True)
    with open(f"{RECORD_DIR}/dev_eval/{target['name'].replace('/', '_')}.jsonl", "w") as f:
        for r, resp, p in zip(rows, resps, per):
            f.write(json.dumps(dict(question=r["question"], response=resp, **p), ensure_ascii=False) + "\n")
    print(f"  {target['name']}: acc={acc:.4f} (n={len(rows)}, {time.time() - t0:.0f}s)")
    return res


DEV_EVAL_RESULTS = globals().get("DEV_EVAL_RESULTS", [])
DEV_ROWS = load_dev_rows() if DEV_EVAL_ENABLED else None
if not DEV_EVAL_ENABLED or not DEV_ROWS:
    print("dev は未確定（DEV_EVAL_ENABLED=False または DEV_EVAL_SOURCE 未設定）。評価と最良設定の選択は行わない。")
    print("準備済み: build_eval_prompt / generate_hf / score_responses / evaluate_target。dev を決めたら DEV_EVAL_* を設定して再実行する。")
    print("注意: 学習データ (Math-CoT-20k) や最終 test をここに指定しない。公開 20,480 行を train/dev に分割しない。")
else:
    assert N_GPUS >= 1
    assert DEV_EVAL_IS_INDEPENDENT, "DEV_EVAL_IS_INDEPENDENT=True を設定する前に、dev が学習データ・最終 test と独立であることを確認する"
    _forbidden = {DATASET_REPO, DATASET_ORIGIN_POOL, os.path.abspath(AO_PARQUET), os.path.abspath(RAW_PARQUET)}
    assert DEV_EVAL_SOURCE not in _forbidden and (not os.path.isfile(DEV_EVAL_SOURCE) or os.path.abspath(DEV_EVAL_SOURCE) not in _forbidden), \
        "学習データ (Math-CoT-20k / OpenR1-Math-220k / AO parquet) を dev にしない"
    print(f"dev: {DEV_EVAL_SOURCE} split={DEV_EVAL_SPLIT} rows={len(DEV_ROWS)}")
    _targets = list(DEV_EVAL_TARGETS)
    if not _targets:
        # このセッションの run の最終 step を候補にする（FSDP checkpoint は verl.model_merger で変換してから評価）
        for _rid, _res in globals().get("TRAIN_RESULTS", {}).items():
            if _res["exit_code"] == 0:
                _st = _res["spec"]["total_steps"]
                _targets.append(dict(name=f"{_rid}@step{_st}", path_or_repo=merge_checkpoint(_rid, _st), subfolder=None, revision=None))
    for _t in _targets:
        DEV_EVAL_RESULTS.append(evaluate_target(_t, DEV_ROWS))
    with open(f"{RECORD_DIR}/dev_eval_results.json", "w") as f:
        json.dump(DEV_EVAL_RESULTS, f, indent=2, ensure_ascii=False)
    print("\n=== dev 集計 ===")
    for _r in sorted(DEV_EVAL_RESULTS, key=lambda x: -x["accuracy"]):
        print(f"  {_r['accuracy']:.4f}  n={_r['n']}  {_r['name']}")
    _search_res = [r for r in DEV_EVAL_RESULTS if "_search" in r["name"] or "_baseline" in r["name"]]
    if _search_res:
        _best = max(_search_res, key=lambda x: x["accuracy"])
        print(f"dev 上の最良: {_best['name']} (acc {_best['accuracy']:.4f})。候補全体を評価し終えるまで『確定』としない。")

## 10. 最終モデルの変換・HF 保存・読み込み確認・Model Card

In [ ]:
# @title 10. 最終モデル: verl.model_merger で HF 形式へ変換 → HF 保存 → 読み込み・短い生成の確認 → Model Card
import json, os, time, torch, glob
from transformers import AutoTokenizer, AutoModelForCausalLM

_final_rid = FINAL_RUN_ID or next(iter(globals().get("TRAIN_RESULTS", {}) or {}), None)
if not _final_rid:
    print("FINAL_RUN_ID が未設定で、このセッションに学習 run も無い。設定セルで FINAL_RUN_ID を指定する")
elif RUN_MODE == "dry_run":
    print("dry_run では変換・アップロードを行わない")
else:
    assert HF_API is not None and HF_CKPT_REPO_ID, "HF にログインしていない"
    _ckpts = list_hf_checkpoints(_final_rid)
    _complete = [c["step"] for c in _ckpts if c["complete"]]
    print(f"{_final_rid}: HF 完了済み steps = {_complete}")
    if FINAL_STEP == "last":
        _tot = _ckpts[-1].get("total_steps") if _ckpts else None
        _step = max(_complete) if _complete else None
        if _tot and _step != _tot:
            print(f"WARN: 最終 step {_tot} の checkpoint が HF に無い（最新は {_step}）。学習が完了していない可能性がある")
    else:
        _step = int(FINAL_STEP)
    assert _step in _complete, f"step {_step} は HF に完了済みでない"
    MERGED_DIR = merge_checkpoint(_final_rid, _step)
    _saved_spec = get_manifest(_final_rid, _step)["run_config"]
    FINAL_REPO = HF_FINAL_MODEL_REPO_ID or f"{HF_ACCOUNT_NAME}/{_final_rid}"
    _sub = f"step{_step}"   # 公開 CoT モデルと同じ stepNNN/ サブディレクトリ構成
    HF_API.create_repo(FINAL_REPO, private=HF_FINAL_MODEL_PRIVATE, repo_type="model", exist_ok=True)
    _existing = [f for f in HF_API.list_repo_files(FINAL_REPO, repo_type="model") if f.startswith(_sub + "/")]
    if _existing:
        raise SystemExit(f"{FINAL_REPO}/{_sub} は既に存在する。上書きしない。HF_FINAL_MODEL_REPO_ID を変える")

    _evals = globals().get("DEV_EVAL_RESULTS", [])
    _eval_md = "\n".join(f"| {e['name']} | {e['dev_source']} ({e['dev_split']}) | {e['n']} | {e['accuracy']:.4f} | {e['scorer']} |" for e in _evals) or "| (未実施) | - | - | - | - |"
    _changes_md = "\n".join(f"| `{c['key']}` | `{c['official']}` | `{c['new']}` | {c['reason']} |" for c in _saved_spec["changes_vs_official"])
    _audit = AO_AUDIT_SUMMARY
    MODEL_CARD = f"""---
license: apache-2.0
base_model: {_saved_spec['base_repo']}
tags: [answer-only-distillation, sft, math, rethink-sft-generalization]
---
# {_final_rid} (Answer-only distillation student, step {_step})

Answer-only (AO) 蒸留の学生モデル。公開 CoT 学生 `{_saved_spec['cot_repo']}/{_saved_spec['cot_subfolder']}` と比較するために、
同じ Base・同じ prompt・同じ問題で、教師 `{TEACHER['repo']}` の**最終回答だけ**を target に Full-parameter SFT した。
学習コードは公式実装 (`verl.trainer.fsdp_sft_trainer_ours`) を無変更で使用。重みは `{_sub}/` にある（公開 CoT モデルと同じ配置）。

## 対応関係と revision
| 役割 | repo | revision |
| --- | --- | --- |
| Teacher | {TEACHER['repo']} | {TEACHER['revision']} |
| Base | {_saved_spec['base_repo']} | {_saved_spec['base_revision']} |
| CoT student (比較対象) | {_saved_spec['cot_repo']} (subfolder {_saved_spec['cot_subfolder']}) | {_saved_spec['cot_revision']} |
| 学習データ | {_saved_spec['dataset_repo']} | {_saved_spec['dataset_revision']} (raw sha256 {DATASET_EXPECTED_SHA256[:12]}, AO parquet sha256 {_saved_spec['data_sha256'][:12]}) |
| コード (Fork) | {FORK_REPO_URL} | {_saved_spec['fork_commit']} (参照元 {UPSTREAM_REFERENCE_COMMIT}) |
| tokenizer | Base と同一（checkpoint 内 huggingface/ から復元） | - |

## AO データの作り方と監査
- target = `response`（教師 CoT+回答）の `</think>` 以降の最後の `\\boxed{{...}}` を原文のまま（style: `{_saved_spec['target_style']}`）。CoT・説明文は含めない。参照正解 `answer` は照合のみ。
- 抽出: `verl/utils/reward_score/math_verify_ours.py` の `last_boxed_only_string`、照合: 同 `compute_score`（math-verify）。
- 行数 {_audit['rows']}、抽出状況 {json.dumps(_audit['extraction_status'])}、照合 {json.dumps(_audit['verify_status'])}、
  複数の異なる box を持つ行 {_audit['rows_multi_distinct_boxes']}、照合不一致で目視確認のうえ保持した行 {AO_ACKNOWLEDGED_VERIFY_MISMATCH_ROWS}。
- 元の 20,480 行・行順・`message`・`advantage` は維持。抽出失敗時の `answer` 代用や行削除はしない。
- prompt は公式と同一で「step by step」の指示を含むが、target は回答のみ。

## 学習設定
- run_id `{_final_rid}`, lr {_saved_spec['lr_str']}, epochs {_saved_spec['epochs']}, global batch {_saved_spec['tbs']}, micro batch/GPU {_saved_spec['micro_bsz']}, GPU 数 {_saved_spec['n_gpus']},
  steps/epoch {_saved_spec['steps_per_epoch']}, total steps {_saved_spec['total_steps']}, warmup {_saved_spec['warmup_steps']}, max_length {_saved_spec['max_length']} (公式 {_saved_spec['official_max_length']}), save_freq {_saved_spec['save_freq']}
- 公式スクリプト `{os.path.basename(_saved_spec['official_script'])}` との差分:

| key | 公式 | 本 run | 理由 |
| --- | --- | --- | --- |
{_changes_md}

- 環境: {json.dumps(ENV_RECORD.get('gpus'))}, torch {PKG_VERSIONS.get('torch')}, transformers {PKG_VERSIONS.get('transformers')}, flash-attn {FLASH_ATTN_VERSION}, Colab={IN_COLAB}
- 再開用 checkpoint（model/optimizer/extra/dataloader 状態）: `{HF_CKPT_REPO_ID}` の `runs/{_final_rid}/global_step_*`

## 評価
| model | dev | n | accuracy | scorer |
| --- | --- | --- | --- | --- |
{_eval_md}

loss の低下だけで能力向上は主張しない。上の表が「未実施」なら性能は未測定。

## 未実施・未確認
- 上記以外のベンチマーク評価は未実施。最良設定の確定は独立 dev で候補を比較してから。
"""
    with open(os.path.join(MERGED_DIR, "..", "README.md"), "w") as f:
        f.write(MODEL_CARD)
    print("uploading merged model to", FINAL_REPO, "/", _sub)
    _info = HF_API.upload_folder(folder_path=MERGED_DIR, path_in_repo=_sub, repo_id=FINAL_REPO, repo_type="model", commit_message=f"{_final_rid} merged step {_step}")
    _info_rd = HF_API.upload_file(path_or_fileobj=os.path.join(MERGED_DIR, "..", "README.md"), path_in_repo="README.md", repo_id=FINAL_REPO, repo_type="model",
                                  commit_message=f"model card for {_final_rid}")
    FINAL_MODEL = dict(repo=FINAL_REPO, subfolder=_sub, revision=_info_rd.oid, weights_commit=_info.oid, url=f"https://huggingface.co/{FINAL_REPO}/tree/{_info_rd.oid}/{_sub}",
                       source_checkpoint=dict(repo=HF_CKPT_REPO_ID, path=f"runs/{_final_rid}/global_step_{_step}"))
    print("uploaded:", FINAL_MODEL)

    # 読み込みと短い生成の確認（アップロード後の revision から）
    _tok = AutoTokenizer.from_pretrained(FINAL_REPO, subfolder=_sub, revision=_info_rd.oid, trust_remote_code=True)
    _mdl = AutoModelForCausalLM.from_pretrained(FINAL_REPO, subfolder=_sub, revision=_info_rd.oid, torch_dtype=torch.bfloat16, trust_remote_code=True).cuda().eval()
    _probe_prompts = [list(RAW_TABLE.column("message")[0].as_py()),
                      [{"role": "user", "content": "What is 17 + 26?\n Please reason step by step, and put your final answer within \\boxed{}."}]]
    GEN_CHECK = []
    for _p in _probe_prompts:
        _enc = _tok.apply_chat_template(_p, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
        with torch.no_grad():
            _g = _mdl.generate(**_enc, max_new_tokens=64, do_sample=False, eos_token_id=_tok.eos_token_id, pad_token_id=_tok.pad_token_id)
        _out = _tok.decode(_g[0, _enc["input_ids"].shape[1]:], skip_special_tokens=False)
        GEN_CHECK.append(dict(prompt=_p[0]["content"][:80], output=_out))
        print("prompt:", _p[0]["content"][:80].replace("\n", " "), "\n  -> output:", repr(_out))
    del _mdl; torch.cuda.empty_cache()
    FINAL_MODEL["generation_check"] = GEN_CHECK
    _own_results = [r for r in globals().get("TRAIN_RESULTS", {}).values() if r["run_id"] == _final_rid]
    write_experiment_record(_saved_spec, _own_results, evaluations=[e for e in _evals if _final_rid in e["name"]] or _evals, final_model=FINAL_MODEL,
                            unverified=(["dev 評価未実施"] if not _evals else []) + ["他ベンチマーク未評価"] +
                                       (["このセッションの学習ログなし（HF の experiment_record を参照）"] if not _own_results else []))